In [ ]:
from pathlib import Path

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
DATA = REPO / "data"

print("=== DIRETÓRIOS RELACIONADOS A MRI ===")

encontrados = []

for pasta in DATA.iterdir():
    if any(k in pasta.name.lower() for k in ["chaos", "mri", "t2"]):
        encontrados.append(pasta)
        print(pasta.relative_to(REPO))

if not encontrados:
    print("Nenhum diretório MRI identificado em data/")

print("\n=== ARQUIVOS DISPONÍVEIS ===")

for pasta in encontrados:
    if not pasta.is_dir():
        continue

    arquivos = [
        p for p in pasta.rglob("*")
        if p.is_file()
    ]

    print(f"\n{pasta.relative_to(REPO)}")
    print("Total de arquivos:", len(arquivos))
    print("NIfTI:", sum(
        p.name.endswith((".nii", ".nii.gz"))
        for p in arquivos
    ))

    for arquivo in arquivos[:12]:
        print(" ", arquivo.relative_to(REPO))

print("\n=== CONFIGURAÇÃO ORIGINAL CHAOS-T2 ===")

for nome in ["config_ssl_upload.py", "backbone.sh"]:
    arquivo = REPO / nome

    if not arquivo.exists():
        continue

    linhas = arquivo.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    indices = set()

    for i, linha in enumerate(linhas):
        if any(t in linha.lower() for t in ["chaost2", "chaos", "mri"]):
            indices.update(
                range(max(0, i - 2), min(len(linhas), i + 5))
            )

    print(f"\n{nome}")

    for i in sorted(indices):
        print(f"{i+1:04}: {linhas[i]}")

In [ ]:
from pathlib import Path
import subprocess
import hashlib
import json
import sys
import os

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
CODIGO = REPO / "exp/eft_mri_chaos_v1"
DESTINO = REPO / "data/CHAOST2/downloads"

CODIGO.mkdir(parents=True, exist_ok=True)
DESTINO.mkdir(parents=True, exist_ok=True)

SCRIPT = CODIGO / "baixar_chaos_urllib.py"

conteudo = r'''
from pathlib import Path
from urllib.request import Request, urlopen
import hashlib
import json
import os
import time

PASTA = Path(
    "/home/1542521/cassio_pereira/ProtoSAM-repro/"
    "data/CHAOST2/downloads"
)

URL = (
    "https://zenodo.org/records/3431873/files/"
    "CHAOS_Train_Sets.zip?download=1"
)

ZIP = PASTA / "CHAOS_Train_Sets.zip"
PARCIAL = PASTA / "CHAOS_Train_Sets.zip.part"
STATUS = PASTA / "status_download.json"

MD5_ESPERADO = "df21053002a1cc86df918a87da3b2c19"
TAMANHO_ESPERADO = 890771694

def registrar(estado, **dados):
    tmp = STATUS.with_suffix(".json.tmp")
    tmp.write_text(json.dumps({
        "estado": estado,
        **dados
    }, indent=2), encoding="utf-8")
    os.replace(tmp, STATUS)

try:
    if not ZIP.exists():
        for tentativa in range(1, 6):
            atual = PARCIAL.stat().st_size if PARCIAL.exists() else 0

            if atual == TAMANHO_ESPERADO:
                break

            try:
                headers = {"User-Agent": "ProtoSAM-Research/1.0"}

                if atual > 0:
                    headers["Range"] = f"bytes={atual}-"

                req = Request(URL, headers=headers)

                with urlopen(req, timeout=90) as resposta:
                    codigo = resposta.status

                    if codigo == 206:
                        intervalo = resposta.headers.get(
                            "Content-Range", ""
                        )
                        assert intervalo.startswith(f"bytes {atual}-"), (
                            f"Resposta Range inesperada: {intervalo}"
                        )
                        modo = "ab"

                    elif codigo == 200:
                        # Se o servidor ignorar Range,
                        # reiniciar a transferência corretamente.
                        modo = "wb"
                        atual = 0

                    else:
                        raise RuntimeError(f"HTTP inesperado: {codigo}")

                    with PARCIAL.open(modo) as arquivo:
                        ultimo_registro = atual

                        while True:
                            bloco = resposta.read(4 * 1024 * 1024)

                            if not bloco:
                                break

                            arquivo.write(bloco)
                            atual += len(bloco)

                            if atual - ultimo_registro >= 64 * 1024 * 1024:
                                registrar(
                                    "baixando",
                                    bytes_baixados=atual,
                                    total_bytes=TAMANHO_ESPERADO,
                                    percentual=round(
                                        100 * atual / TAMANHO_ESPERADO, 2
                                    ),
                                    tentativa=tentativa
                                )
                                print(
                                    f"Download: "
                                    f"{100 * atual / TAMANHO_ESPERADO:.1f}%",
                                    flush=True
                                )
                                ultimo_registro = atual

                if PARCIAL.stat().st_size == TAMANHO_ESPERADO:
                    break

                raise RuntimeError("Transferência incompleta")

            except Exception as e:
                print(
                    f"Tentativa {tentativa}/5 falhou: {e}",
                    flush=True
                )

                if tentativa == 5:
                    raise

                time.sleep(5)

        arquivo = PARCIAL
    else:
        arquivo = ZIP

    registrar("verificando_md5")

    assert arquivo.stat().st_size == TAMANHO_ESPERADO, (
        "Tamanho do arquivo não corresponde ao original."
    )

    h = hashlib.md5()

    with arquivo.open("rb") as f:
        for bloco in iter(
            lambda: f.read(4 * 1024 * 1024), b""
        ):
            h.update(bloco)

    digest = h.hexdigest()

    assert digest == MD5_ESPERADO, (
        f"MD5 incorreto: {digest}"
    )

    if arquivo == PARCIAL:
        os.replace(PARCIAL, ZIP)

    registrar(
        "concluido",
        arquivo=str(ZIP),
        tamanho_bytes=ZIP.stat().st_size,
        md5=digest
    )

    print("DOWNLOAD CONCLUÍDO E MD5 VERIFICADO", flush=True)

except Exception as e:
    registrar("erro", mensagem=str(e))
    raise
'''

compile(conteudo, str(SCRIPT), "exec")

if SCRIPT.exists():
    assert SCRIPT.read_text(encoding="utf-8") == conteudo
else:
    SCRIPT.write_text(conteudo, encoding="utf-8")

LOG = DESTINO / "download_urllib.log"
LANCAMENTO = DESTINO / "lancamento_urllib.json"

assert not LANCAMENTO.exists(), "Download já lançado"
assert not LOG.exists(), "Log anterior encontrado"

fd = os.open(
    str(LANCAMENTO),
    os.O_CREAT | os.O_EXCL | os.O_WRONLY,
    0o600
)

try:
    with LOG.open("x") as log:
        processo = subprocess.Popen(
            [sys.executable, "-u", str(SCRIPT)],
            cwd=str(REPO),
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True
        )

    with os.fdopen(fd, "w") as f:
        json.dump({
            "pid": processo.pid,
            "script": str(SCRIPT),
            "modo": "detached"
        }, f, indent=2)

except Exception:
    try:
        os.close(fd)
    except OSError:
        pass
    LANCAMENTO.unlink(missing_ok=True)
    raise

print("=== DOWNLOAD CHAOS MRI ===")
print("PID:", processo.pid)
print("Método: Python urllib")
print("Origem: Zenodo oficial")
print("Arquivo:", DESTINO / "CHAOS_Train_Sets.zip")
print("Log:", LOG)
print("Modo: detached")
print("Lançamento solicitado; download ainda não verificado")

In [ ]:
from pathlib import Path
import json
import zipfile

PASTA = Path(
    "/home/1542521/cassio_pereira/ProtoSAM-repro/"
    "data/CHAOST2/downloads"
)

ZIP = PASTA / "CHAOS_Train_Sets.zip"
PARCIAL = PASTA / "CHAOS_Train_Sets.zip.part"
STATUS = PASTA / "status_download.json"
LANCAMENTO = PASTA / "lancamento_urllib.json"
LOG = PASTA / "download_urllib.log"

TAMANHO_ESPERADO = 890771694

def ler_json(arquivo):
    if not arquivo.exists():
        return {}
    try:
        return json.loads(arquivo.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return {}

registro = ler_json(LANCAMENTO)
status = ler_json(STATUS)

pid = registro.get("pid")
ativo = False

if pid:
    proc = Path(f"/proc/{pid}")
    try:
        cmd = (proc / "cmdline").read_bytes()
        estado_proc = (proc / "stat").read_text().split(") ")[1][0]
        ativo = (
            b"baixar_chaos_urllib.py" in cmd
            and estado_proc not in ("Z", "X")
        )
    except (OSError, IndexError):
        pass

arquivo_atual = ZIP if ZIP.exists() else PARCIAL
bytes_atuais = (
    arquivo_atual.stat().st_size
    if arquivo_atual.exists() else 0
)

percentual = min(100, bytes_atuais / TAMANHO_ESPERADO * 100)

print("=" * 58)
print("DOWNLOAD CHAOS MRI — ZENODO")
print("=" * 58)

print("PID:", pid)
print("Processo ativo:", "SIM" if ativo else "NÃO")
print("Estado:", status.get("estado", "inicializando"))

print(f"\nBaixado: {bytes_atuais / 1024**2:.1f} MiB")
print(f"Esperado: {TAMANHO_ESPERADO / 1024**2:.1f} MiB")
print(f"Progresso: {percentual:.2f}%")

blocos = int(percentual * 30 / 100)
print("[" + "█" * blocos + "░" * (30 - blocos) + "]")

print("\n=== INTEGRIDADE ===")

if status.get("estado") == "concluido":
    print("MD5 verificado:", status.get("md5"))
    print("ZIP encontrado:", ZIP.is_file())
    print(
        "Estrutura ZIP reconhecida:",
        zipfile.is_zipfile(ZIP)
    )
elif status.get("estado") == "verificando_md5":
    print("Download recebido; verificando MD5.")
else:
    print("Integridade ainda não confirmada.")

print("\n=== DIAGNÓSTICO ===")

if status.get("estado") == "erro":
    print("ERRO:", status.get("mensagem", "não especificado"))
elif status.get("estado") == "concluido":
    print("DOWNLOAD CONCLUÍDO")
elif ativo:
    print("DOWNLOAD EM EXECUÇÃO")
else:
    print("Processo inativo sem conclusão confirmada.")

if LOG.exists():
    print("\n=== ÚLTIMAS MENSAGENS ===")
    with LOG.open(encoding="utf-8", errors="replace") as f:
        linhas = f.readlines()

    for linha in linhas[-12:]:
        print(linha.rstrip())

In [ ]:
from pathlib import Path
from collections import Counter
import zipfile

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

ZIP = (
    REPO / "data/CHAOST2/downloads/CHAOS_Train_Sets.zip"
)

assert ZIP.is_file()
assert zipfile.is_zipfile(ZIP)

with zipfile.ZipFile(ZIP) as z:
    arquivos = [
        item for item in z.infolist()
        if not item.is_dir()
    ]

    caminhos = [item.filename for item in arquivos]

    # Segurança antes de uma futura extração
    assert all(
        not Path(p).is_absolute()
        and ".." not in Path(p).parts
        for p in caminhos
    )

    print("=== 1. VISÃO GERAL ===")
    print("Total de arquivos:", len(arquivos))
    print(
        "Tamanho descompactado:",
        round(sum(i.file_size for i in arquivos) / 1024**3, 2),
        "GiB"
    )

    print("\n=== 2. DIRETÓRIOS PRINCIPAIS ===")

    grupos = Counter(
        "/".join(p.split("/")[:3])
        for p in caminhos
    )

    for pasta, quantidade in grupos.most_common(15):
        print(f"{quantidade:6d}  {pasta}")

    # Identificar os arquivos relacionados à MRI
    mr = [
        p for p in caminhos
        if "MR" in p.upper().split("/")
    ]

    t2 = [
        p for p in mr
        if "T2SPIR" in p.upper().replace("_", "").replace("-", "")
    ]

    print("\n=== 3. MRI ===")
    print("Arquivos relacionados a MR:", len(mr))
    print("Arquivos relacionados a T2-SPIR:", len(t2))

    print("\n=== 4. ESTRUTURA T2-SPIR ===")

    grupos_t2 = Counter(
        "/".join(p.split("/")[:7])
        for p in t2
    )

    for pasta, quantidade in grupos_t2.most_common(20):
        print(f"{quantidade:5d}  {pasta}")

    print("\n=== 5. EXEMPLOS DE IMAGENS E ANOTAÇÕES ===")

    for caminho in t2[:25]:
        print(caminho)

    print("\n=== 6. EXTENSÕES ENCONTRADAS NO MRI ===")

    extensoes = Counter(
        Path(p).suffix.lower() or "(sem extensão)"
        for p in mr
    )

    for extensao, quantidade in extensoes.most_common():
        print(f"{extensao:15s} {quantidade}")

    print("\n=== 7. OUTROS EXEMPLOS DE ARQUIVOS MRI ===")

    for caminho in mr[:12]:
        print(caminho)

print("\nInspeção concluída. Nenhum arquivo foi extraído.")

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import zipfile
import shutil
import json
import os

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

ZIP = REPO / "data/CHAOST2/downloads/CHAOS_Train_Sets.zip"
DESTINO = REPO / "data/CHAOST2/RawData"
MANIFESTO = DESTINO / "manifesto_t2spir.json"

assert ZIP.is_file()
DESTINO.mkdir(parents=True, exist_ok=True)

inventario = []
extraidos = 0
existentes = 0

with zipfile.ZipFile(ZIP) as arquivo:
    selecionados = []

    for item in arquivo.infolist():
        if item.is_dir():
            continue

        partes = Path(item.filename).parts

        if (
            len(partes) >= 6
            and partes[0] == "Train_Sets"
            and partes[1] == "MR"
            and partes[3].upper() == "T2SPIR"
        ):
            selecionados.append(item)

    assert selecionados, "Nenhum arquivo T2-SPIR encontrado."

    print("=== EXTRAÇÃO CHAOS T2-SPIR ===")
    print("Arquivos selecionados:", len(selecionados))
    print(
        "Tamanho descompactado:",
        round(sum(i.file_size for i in selecionados) / 1024**2, 1),
        "MiB"
    )

    for item in selecionados:
        partes = Path(item.filename).parts
        paciente = partes[2]

        destino = DESTINO / item.filename

        # Impedir caminhos fora do diretório de destino.
        assert destino.resolve().is_relative_to(DESTINO.resolve())

        destino.parent.mkdir(parents=True, exist_ok=True)

        if destino.exists():
            assert destino.stat().st_size == item.file_size, (
                f"Arquivo existente com tamanho diferente: {destino}"
            )
            existentes += 1
        else:
            temporario = destino.with_name(destino.name + ".part")

            with arquivo.open(item) as origem:
                with temporario.open("wb") as saida:
                    shutil.copyfileobj(origem, saida)

            assert temporario.stat().st_size == item.file_size
            os.replace(temporario, destino)
            extraidos += 1

        inventario.append({
            "paciente": paciente,
            "caminho": item.filename,
            "tamanho_bytes": item.file_size,
            "crc32": item.CRC,
            "tipo": Path(item.filename).suffix.lower(),
            "subpasta": partes[4]
        })

# Salvar inventário
manifesto = {
    "dataset": "CHAOS",
    "modalidade": "MRI",
    "sequencia": "T2-SPIR",
    "origem": str(ZIP),
    "diretorio": str(DESTINO),
    "arquivos": inventario
}

temporario = MANIFESTO.with_suffix(".json.tmp")
temporario.write_text(
    json.dumps(manifesto, indent=2, ensure_ascii=False),
    encoding="utf-8"
)
os.replace(temporario, MANIFESTO)

# Resumo por paciente
por_paciente = defaultdict(Counter)

for item in inventario:
    por_paciente[item["paciente"]][
        (item["subpasta"], item["tipo"])
    ] += 1

print("\n=== RESULTADO ===")
print("Arquivos extraídos agora:", extraidos)
print("Arquivos já existentes:", existentes)
print("Total:", len(inventario))
print("Pacientes:", len(por_paciente))

print("\n=== DISTRIBUIÇÃO POR PACIENTE ===")

for paciente in sorted(por_paciente, key=lambda x: int(x)):
    grupos = por_paciente[paciente]
    descricao = ", ".join(
        f"{subpasta}/{ext}: {quantidade}"
        for (subpasta, ext), quantidade in sorted(grupos.items())
    )
    print(f"Paciente {paciente:>3}: {descricao}")

print("\nDiretório:", DESTINO)
print("Manifesto:", MANIFESTO)
print("Extração concluída.")

In [ ]:
from pathlib import Path
from collections import Counter
from PIL import Image
import SimpleITK as sitk
import numpy as np

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

MRI = (
    REPO / "data/CHAOST2/RawData/"
    "Train_Sets/MR"
)

pacientes = sorted(
    [p for p in MRI.iterdir() if p.is_dir()],
    key=lambda p: int(p.name)
)

assert len(pacientes) == 20

total_pares = 0
modos_png = Counter()
valores_mascaras = set()

print("=== PAREAMENTO MRI T2-SPIR ===")
print("Paciente | DICOM | PNG | Pares | Dimensão DICOM | Modo PNG")
print("-" * 73)

for paciente in pacientes:
    pasta = paciente / "T2SPIR"

    dicoms = {
        p.stem: p
        for p in (pasta / "DICOM_anon").glob("*.dcm")
    }

    mascaras = {
        p.stem: p
        for p in (pasta / "Ground").glob("*.png")
    }

    # Comparação pelos nomes dos arquivos, não
    # apenas pela quantidade de fatias.
    faltando_png = set(dicoms) - set(mascaras)
    faltando_dicom = set(mascaras) - set(dicoms)

    assert not faltando_png, (
        f"Paciente {paciente.name}: PNG ausente: {faltando_png}"
    )
    assert not faltando_dicom, (
        f"Paciente {paciente.name}: DICOM ausente: {faltando_dicom}"
    )

    nomes = sorted(dicoms)
    total_pares += len(nomes)

    # Amostra DICOM do centro do volume
    arquivo_dicom = dicoms[nomes[len(nomes) // 2]]
    imagem_dicom = sitk.GetArrayFromImage(
        sitk.ReadImage(str(arquivo_dicom))
    )

    # Conferir todas as máscaras deste paciente
    for nome in nomes:
        with Image.open(mascaras[nome]) as im:
            modos_png[im.mode] += 1
            arr = np.asarray(im)

            if arr.ndim == 2:
                valores_mascaras.update(
                    int(v) for v in np.unique(arr)
                )

    with Image.open(mascaras[nomes[len(nomes) // 2]]) as im:
        modo = im.mode
        dimensoes_png = im.size

    # DICOM geralmente é 2D ou (1,H,W)
    dimensoes_dicom = imagem_dicom.shape[-2:]

    assert tuple(dimensoes_png[::-1]) == tuple(dimensoes_dicom), (
        f"Dimensões incompatíveis: paciente {paciente.name}"
    )

    print(
        f"{paciente.name:>8} | "
        f"{len(dicoms):5d} | "
        f"{len(mascaras):3d} | "
        f"{len(nomes):5d} | "
        f"{str(dimensoes_dicom):>15} | "
        f"{modo}"
    )

assert total_pares == 623

print("\n=== RESUMO ===")
print("Pacientes:", len(pacientes))
print("Pares DICOM/PNG validados:", total_pares)
print("Modos PNG:", dict(modos_png))

if valores_mascaras:
    print("Valores encontrados nas máscaras:", sorted(valores_mascaras))
else:
    print("Máscaras multicanais: inspecionar codificação RGB.")

print("\n=== EXEMPLO DO PRIMEIRO PACIENTE ===")

primeiro = pacientes[0] / "T2SPIR"
arquivo_dicom = sorted(
    (primeiro / "DICOM_anon").glob("*.dcm")
)[0]

arquivo_png = primeiro / "Ground" / (
    arquivo_dicom.stem + ".png"
)

imagem = sitk.GetArrayFromImage(
    sitk.ReadImage(str(arquivo_dicom))
)

with Image.open(arquivo_png) as im:
    mascara = np.asarray(im)
    modo = im.mode

print("DICOM:", arquivo_dicom.name)
print("PNG:", arquivo_png.name)
print("Intensidade MRI:", float(imagem.min()), "até", float(imagem.max()))
print("Formato da imagem:", imagem.shape)
print("Formato da máscara:", mascara.shape)
print("Modo da máscara:", modo)
print("Valores únicos:", np.unique(mascara)[:30])

In [ ]:
from pathlib import Path
import json
import re

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

NOTEBOOK = REPO / "data/data_processing.ipynb"

assert NOTEBOOK.is_file(), "Notebook original não encontrado"

nb = json.loads(NOTEBOOK.read_text(encoding="utf-8"))
celulas = nb["cells"]

print("=== 1. CÉLULAS DE PRÉ-PROCESSAMENTO MRI ===")

padrao = re.compile(
    r"CHAOS|CHAOST2|T2SPIR|MRI|MR_T2|"
    r"normalize_mr|normaliz.*mr",
    re.IGNORECASE
)

indices = [
    i for i, c in enumerate(celulas)
    if padrao.search("".join(c.get("source", [])))
]

print("Células relevantes:", indices)

# Incluir a célula seguinte para capturar código
# que continua após uma descrição ou configuração.
selecionadas = sorted(set(
    j
    for i in indices
    for j in (i, i + 1)
    if j < len(celulas)
))

for i in selecionadas[:14]:
    c = celulas[i]
    fonte = "".join(c.get("source", []))
    linhas = fonte.splitlines()

    print(f"\n--- Célula original {i} ({c['cell_type']}) ---")

    for numero, linha in enumerate(linhas[:65], 1):
        print(f"{numero:03d}: {linha}")

    if len(linhas) > 65:
        print(f"... mais {len(linhas) - 65} linhas")

print("\n=== 2. MAPEAMENTO DE CLASSES E CONFIGURAÇÃO ===")

arquivos = [
    REPO / "dataloaders/GenericSuperDatasetv2.py",
    REPO / "dataloaders/ManualAnnoDatasetv2.py",
    REPO / "config_ssl_upload.py",
]

termos = [
    "CHAOST2",
    "LABEL_NAME",
    "LABEL_VALUE",
    "label_values",
    "real_label_name",
    "label_sets",
    "exclude_cls_list",
]

for arquivo in arquivos:
    if not arquivo.exists():
        continue

    linhas = arquivo.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    encontrados = set()

    for i, linha in enumerate(linhas):
        if any(t.lower() in linha.lower() for t in termos):
            encontrados.update(
                range(
                    max(0, i - 2),
                    min(len(linhas), i + 5)
                )
            )

    print(f"\n--- {arquivo.relative_to(REPO)} ---")

    for i in sorted(encontrados):
        print(f"{i+1:04d}: {linhas[i]}")

print("\nInspeção concluída. Nenhum arquivo modificado.")

In [ ]:
from pathlib import Path
import json

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
DATA = REPO / "data"

print("=== 1. SCRIPT ORIGINAL DE CONVERSÃO ===")

candidatos = [
    DATA / "dcm_img_to_nii.sh",
    REPO / "dcm_img_to_nii.sh",
]

scripts = [p for p in candidatos if p.is_file()]

if not scripts:
    scripts = list(DATA.glob("*dcm*nii*.sh"))

if scripts:
    for arquivo in scripts:
        print(f"\nArquivo: {arquivo.relative_to(REPO)}")
        linhas = arquivo.read_text(
            encoding="utf-8", errors="replace"
        ).splitlines()

        for i, linha in enumerate(linhas, 1):
            print(f"{i:04}: {linha}")
else:
    print("Script de conversão não encontrado.")

print("\n=== 2. OUTROS SCRIPTS DE PREPARAÇÃO ===")

for arquivo in sorted(DATA.glob("*.sh")):
    print(arquivo.relative_to(REPO))

print("\n=== 3. CÓDIGO DE CONVERSÃO NO NOTEBOOK ORIGINAL ===")

nb = json.loads(
    (DATA / "data_processing.ipynb").read_text(
        encoding="utf-8"
    )
)

termos = [
    "dcm_img_to_nii",
    "dicom",
    "ground",
    "t2spir",
    "63",
    "label_",
]

for indice, celula in enumerate(nb["cells"][:15]):
    fonte = "".join(celula.get("source", []))

    if not any(t in fonte.lower() for t in termos):
        continue

    print(f"\n--- Célula {indice} ---")

    linhas = fonte.splitlines()

    for numero, linha in enumerate(linhas[:90], 1):
        print(f"{numero:03}: {linha}")

    if len(linhas) > 90:
        print(f"... {len(linhas) - 90} linhas adicionais")

print("\nInspeção concluída. Nenhum arquivo modificado.")

In [ ]:
from pathlib import Path
import numpy as np
import SimpleITK as sitk

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
MRI = REPO / "data/CHAOST2/RawData/Train_Sets/MR"

pacientes = sorted(
    [p for p in MRI.iterdir() if p.is_dir()],
    key=lambda p: int(p.name)
)

def metadados_dicom(arquivo):
    leitor = sitk.ImageFileReader()
    leitor.SetFileName(str(arquivo))
    leitor.ReadImageInformation()

    def tag(chave):
        if leitor.HasMetaDataKey(chave):
            return leitor.GetMetaData(chave).strip()
        return None

    def vetor(chave):
        valor = tag(chave)
        if valor is None:
            return None
        return np.array(
            [float(x) for x in valor.split("\\")],
            dtype=float
        )

    return {
        "posicao": vetor("0020|0032"),
        "orientacao": vetor("0020|0037"),
        "pixel_spacing": vetor("0028|0030"),
        "instance": tag("0020|0013"),
        "slice_thickness": tag("0018|0050"),
    }

print("=== GEOMETRIA CHAOS T2-SPIR ===")
print(
    "Paciente | Fatias | Pixel spacing | "
    "Espaçamento Z | Ordem física"
)
print("-" * 78)

total = 0

for paciente in pacientes:
    pasta = paciente / "T2SPIR/DICOM_anon"
    arquivos = sorted(pasta.glob("*.dcm"))

    assert arquivos, f"Sem DICOM: {paciente.name}"

    dados = [metadados_dicom(p) for p in arquivos]
    total += len(arquivos)

    completos = all(
        d["posicao"] is not None
        and d["orientacao"] is not None
        and d["pixel_spacing"] is not None
        for d in dados
    )

    if not completos:
        print(
            f"{paciente.name:>8} | {len(arquivos):6d} | "
            "METADADOS INCOMPLETOS"
        )
        continue

    orientacoes = np.array([
        d["orientacao"] for d in dados
    ])

    # A orientação precisa permanecer constante no volume.
    assert np.allclose(
        orientacoes, orientacoes[0], atol=1e-4
    ), f"Orientação inconsistente no paciente {paciente.name}"

    orientacao = orientacoes[0]

    eixo_linha = orientacao[:3]
    eixo_coluna = orientacao[3:]
    normal = np.cross(eixo_linha, eixo_coluna)

    # Posição física ao longo do eixo perpendicular
    # ao plano das imagens.
    posicoes = np.array([
        np.dot(d["posicao"], normal)
        for d in dados
    ])

    diferencas = np.diff(posicoes)

    if np.all(diferencas > 0):
        ordem = "crescente"
    elif np.all(diferencas < 0):
        ordem = "decrescente"
    else:
        ordem = "IRREGULAR"

    espacamento_z = (
        float(np.median(np.abs(diferencas)))
        if len(diferencas) else float("nan")
    )

    pixel_spacing = dados[0]["pixel_spacing"]

    # Detectar inconsistências de espaçamento dentro do paciente
    assert all(
        np.allclose(d["pixel_spacing"], pixel_spacing)
        for d in dados
    )

    print(
        f"{paciente.name:>8} | "
        f"{len(arquivos):6d} | "
        f"{pixel_spacing[0]:.3f} × {pixel_spacing[1]:.3f} | "
        f"{espacamento_z:8.3f} mm | "
        f"{ordem}"
    )

print("\n=== RESUMO ===")
print("Pacientes:", len(pacientes))
print("Fatias DICOM examinadas:", total)
print("Esperadas:", 623)

assert len(pacientes) == 20
assert total == 623

print("\nVerificação concluída. Nenhum arquivo modificado.")

In [ ]:
from pathlib import Path
from PIL import Image
import SimpleITK as sitk
import numpy as np
import json
import os

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

ORIGEM = (
    REPO / "data/CHAOST2/RawData/Train_Sets/MR"
)

DESTINO = REPO / "data/CHAOST2/niis/T2SPIR"
DESTINO.mkdir(parents=True, exist_ok=True)

VALORES = np.array([0, 63, 126, 189, 252])

pacientes = sorted(
    [p for p in ORIGEM.iterdir() if p.is_dir()],
    key=lambda p: int(p.name)
)

assert len(pacientes) == 20

def ler_metadados(arquivo):
    leitor = sitk.ImageFileReader()
    leitor.SetFileName(str(arquivo))
    leitor.ReadImageInformation()

    def vetor(tag):
        valor = leitor.GetMetaData(tag)
        return np.array(
            [float(x) for x in valor.split("\\")],
            dtype=np.float64
        )

    return (
        vetor("0020|0032"),  # ImagePositionPatient
        vetor("0020|0037"),  # ImageOrientationPatient
        vetor("0028|0030"),  # PixelSpacing
    )

def salvar_nifti(obj, destino):
    if destino.exists():
        atual = sitk.ReadImage(str(destino))

        assert atual.GetSize() == obj.GetSize()
        assert np.allclose(
            atual.GetSpacing(), obj.GetSpacing(), atol=1e-4
        )
        assert np.allclose(
            atual.GetOrigin(), obj.GetOrigin(), atol=1e-4
        )
        assert np.allclose(
            atual.GetDirection(), obj.GetDirection(), atol=1e-4
        )
        assert np.array_equal(
            sitk.GetArrayFromImage(atual),
            sitk.GetArrayFromImage(obj)
        )
        return "verificado"

    temporario = destino.with_name(
        destino.name.replace(".nii.gz", ".tmp.nii.gz")
    )

    sitk.WriteImage(obj, str(temporario), True)
    os.replace(temporario, destino)
    return "criado"

manifesto = []
total_fatias = 0

print("=== CONVERSÃO CHAOS T2-SPIR ===")
print("Índice | ID original | Fatias | Shape | Spacing Z | Status")
print("-" * 77)

for indice, paciente in enumerate(pacientes):

    pasta = paciente / "T2SPIR"

    dicoms = sorted(
        (pasta / "DICOM_anon").glob("*.dcm")
    )

    assert dicoms

    imagens = []
    mascaras = []
    posicoes = []
    orientacoes = []
    espacamentos = []

    for arquivo in dicoms:
        png = pasta / "Ground" / f"{arquivo.stem}.png"
        assert png.is_file(), png

        imagem = sitk.GetArrayFromImage(
            sitk.ReadImage(str(arquivo))
        )[0]

        with Image.open(png) as im:
            mascara_original = np.asarray(im).copy()

        assert imagem.shape == mascara_original.shape
        assert np.isin(mascara_original, VALORES).all()

        # 0,63,126,189,252 -> 0,1,2,3,4
        mascara = (mascara_original // 63).astype(np.uint8)

        posicao, orientacao, pixel_spacing = ler_metadados(arquivo)

        imagens.append(imagem.astype(np.float32))
        mascaras.append(mascara)
        posicoes.append(posicao)
        orientacoes.append(orientacao)
        espacamentos.append(pixel_spacing)

    posicoes = np.stack(posicoes)
    orientacoes = np.stack(orientacoes)
    espacamentos = np.stack(espacamentos)

    assert np.allclose(
        orientacoes, orientacoes[0], atol=1e-4
    )
    assert np.allclose(
        espacamentos, espacamentos[0], atol=1e-4
    )

    orientacao = orientacoes[0]

    eixo_x = orientacao[:3]
    eixo_y = orientacao[3:]
    eixo_z = np.cross(eixo_x, eixo_y)
    eixo_z /= np.linalg.norm(eixo_z)

    matriz_direcao = np.column_stack([
        eixo_x, eixo_y, eixo_z
    ])

    assert np.allclose(
        matriz_direcao.T @ matriz_direcao,
        np.eye(3),
        atol=1e-3
    )

    # Espaçamento efetivo calculado pelas posições DICOM
    coordenadas_z = posicoes @ eixo_z
    distancias = np.diff(coordenadas_z)

    assert np.all(distancias > 0), (
        f"Ordem física incorreta no paciente {paciente.name}"
    )

    spacing_z = float(np.median(distancias))

    # Confirmar que as fatias formam uma grade física regular
    posicoes_esperadas = (
        posicoes[0][None, :]
        + np.arange(len(dicoms))[:, None]
        * spacing_z
        * eixo_z[None, :]
    )

    erro_geometrico = float(np.max(
        np.linalg.norm(posicoes - posicoes_esperadas, axis=1)
    ))

    assert erro_geometrico < 0.5, (
        f"Geometria irregular no paciente {paciente.name}: "
        f"{erro_geometrico:.3f} mm"
    )

    # DICOM PixelSpacing = [linha, coluna]
    spacing = (
        float(espacamentos[0][1]),
        float(espacamentos[0][0]),
        spacing_z
    )

    origem_fisica = tuple(float(x) for x in posicoes[0])
    direcao = tuple(float(x) for x in matriz_direcao.ravel())

    array_img = np.stack(imagens)
    array_lb = np.stack(mascaras)

    assert array_img.shape == array_lb.shape

    img_obj = sitk.GetImageFromArray(array_img)
    lb_obj = sitk.GetImageFromArray(array_lb)

    for obj in (img_obj, lb_obj):
        obj.SetSpacing(spacing)
        obj.SetOrigin(origem_fisica)
        obj.SetDirection(direcao)

    caminho_img = DESTINO / f"image_{indice}.nii.gz"
    caminho_lb = DESTINO / f"label_{indice}.nii.gz"

    status_img = salvar_nifti(img_obj, caminho_img)
    status_lb = salvar_nifti(lb_obj, caminho_lb)

    total_fatias += len(dicoms)

    manifesto.append({
        "indice": indice,
        "paciente_original": int(paciente.name),
        "fatias": len(dicoms),
        "shape_zyx": list(array_img.shape),
        "spacing_xyz": list(spacing),
        "erro_geometrico_max_mm": erro_geometrico,
        "imagem": caminho_img.name,
        "mascara": caminho_lb.name,
        "classes": sorted(int(x) for x in np.unique(array_lb)),
        "arquivos_dicom": [p.name for p in dicoms]
    })

    print(
        f"{indice:6d} | "
        f"{paciente.name:11s} | "
        f"{len(dicoms):6d} | "
        f"{str(array_img.shape):18s} | "
        f"{spacing_z:9.3f} | "
        f"{status_img}/{status_lb}"
    )

assert total_fatias == 623

manifesto_path = DESTINO / "manifesto_conversao.json"

conteudo = json.dumps(
    {
        "dataset": "CHAOS",
        "sequencia": "T2-SPIR",
        "conversao": "DICOM e PNG para NIfTI via SimpleITK",
        "classes": {
            "0": "BG",
            "1": "LIVER",
            "2": "RK",
            "3": "LK",
            "4": "SPLEEN"
        },
        "pacientes": manifesto
    },
    indent=2,
    ensure_ascii=False
)

if manifesto_path.exists():
    assert manifesto_path.read_text(encoding="utf-8") == conteudo
else:
    temp = manifesto_path.with_suffix(".json.tmp")
    temp.write_text(conteudo, encoding="utf-8")
    os.replace(temp, manifesto_path)

print("\n=== RESUMO ===")
print("Pacientes convertidos:", len(manifesto))
print("Fatias convertidas:", total_fatias)
print("Volumes de imagem:", len(list(DESTINO.glob("image_*.nii.gz"))))
print("Volumes de máscara:", len(list(DESTINO.glob("label_*.nii.gz"))))
print("Valores das classes: 0, 1, 2, 3, 4")
print("Diretório:", DESTINO)
print("Manifesto:", manifesto_path)

In [ ]:
from pathlib import Path
import SimpleITK as sitk
import numpy as np
import json
import os

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

ORIGEM = REPO / "data/CHAOST2/niis/T2SPIR"
DESTINO = REPO / "data/CHAOST2/chaos_MR_T2_normalized"

DESTINO.mkdir(parents=True, exist_ok=True)

SPACING_ALVO = (1.25, 1.25, 7.70)
RESOLUCAO = 256
PERCENTIL = 99.5

def salvar_seguro(imagem, caminho):
    if caminho.exists():
        anterior = sitk.ReadImage(str(caminho))
        assert anterior.GetSize() == imagem.GetSize()
        assert np.allclose(
            anterior.GetSpacing(), imagem.GetSpacing()
        )
        assert np.allclose(
            anterior.GetOrigin(), imagem.GetOrigin()
        )
        assert np.allclose(
            anterior.GetDirection(), imagem.GetDirection()
        )
        assert np.allclose(
            sitk.GetArrayFromImage(anterior),
            sitk.GetArrayFromImage(imagem),
            atol=1e-5
        )
        return "verificado"

    temporario = caminho.with_name(
        caminho.name.replace(".nii.gz", ".tmp.nii.gz")
    )

    sitk.WriteImage(imagem, str(temporario), True)
    os.replace(temporario, caminho)
    return "criado"


def criar_referencia(imagem):
    tamanho_original = np.array(imagem.GetSize())
    spacing_original = np.array(imagem.GetSpacing())

    # Preservar toda a extensão física, sem o voxel
    # adicional artificial da implementação original.
    tamanho_novo = (
        np.ceil(
            (tamanho_original - 1)
            * spacing_original
            / np.array(SPACING_ALVO)
        ).astype(int) + 1
    )

    referencia = sitk.Image(
        [int(x) for x in tamanho_novo],
        sitk.sitkFloat32
    )

    referencia.SetSpacing(SPACING_ALVO)
    referencia.SetOrigin(imagem.GetOrigin())
    referencia.SetDirection(imagem.GetDirection())

    return referencia


def reamostrar_mascara_por_classe(mascara, referencia):
    original = sitk.GetArrayFromImage(mascara)

    assert np.isin(original, [0, 1, 2, 3, 4]).all()

    shape = tuple(reversed(referencia.GetSize()))
    saida = np.zeros(shape, dtype=np.uint8)

    for classe in range(1, 5):
        binaria = sitk.GetImageFromArray(
            (original == classe).astype(np.float32)
        )
        binaria.CopyInformation(mascara)

        interpolada = sitk.Resample(
            binaria,
            referencia,
            sitk.Transform(),
            sitk.sitkLinear,
            0.0,
            sitk.sitkFloat32
        )

        valores = sitk.GetArrayFromImage(interpolada)

        # Equivalente à lógica original:
        # interpolação linear e np.rint por classe.
        saida[np.rint(valores) == 1] = classe

    resultado = sitk.GetImageFromArray(saida)
    resultado.CopyInformation(referencia)

    return resultado


manifesto = []
total_fatias = 0

print("=== PRÉ-PROCESSAMENTO CHAOS-T2 ===")
print("Paciente | Original Z | Final Z | Resolução | Intensidade máxima")
print("-" * 77)

for pid in range(20):
    caminho_imagem = ORIGEM / f"image_{pid}.nii.gz"
    caminho_mascara = ORIGEM / f"label_{pid}.nii.gz"

    assert caminho_imagem.is_file()
    assert caminho_mascara.is_file()

    imagem = sitk.ReadImage(str(caminho_imagem))
    mascara = sitk.ReadImage(str(caminho_mascara))

    assert imagem.GetSize() == mascara.GetSize()
    assert np.allclose(
        imagem.GetSpacing(), mascara.GetSpacing()
    )
    assert np.allclose(
        imagem.GetOrigin(), mascara.GetOrigin()
    )
    assert np.allclose(
        imagem.GetDirection(), mascara.GetDirection()
    )

    # 1. Limitar intensidades extremas
    array = sitk.GetArrayFromImage(imagem).astype(np.float32)

    limite = float(np.percentile(array, PERCENTIL))
    array = np.minimum(array, limite)

    imagem_limitada = sitk.GetImageFromArray(array)
    imagem_limitada.CopyInformation(imagem)

    # 2. Reamostragem espacial
    referencia = criar_referencia(imagem_limitada)

    imagem_res = sitk.Resample(
        imagem_limitada,
        referencia,
        sitk.Transform(),
        sitk.sitkLinear,
        float(array.min()),
        sitk.sitkFloat32
    )

    mascara_res = reamostrar_mascara_por_classe(
        mascara, referencia
    )

    assert imagem_res.GetSize() == mascara_res.GetSize()

    # 3. Recorte central de 256 x 256
    largura, altura, profundidade = imagem_res.GetSize()

    assert largura >= RESOLUCAO
    assert altura >= RESOLUCAO

    inicio_x = (largura - RESOLUCAO) // 2
    inicio_y = (altura - RESOLUCAO) // 2

    tamanho_roi = [
        RESOLUCAO,
        RESOLUCAO,
        profundidade
    ]

    inicio_roi = [inicio_x, inicio_y, 0]

    imagem_final = sitk.RegionOfInterest(
        imagem_res, tamanho_roi, inicio_roi
    )

    mascara_final = sitk.RegionOfInterest(
        mascara_res, tamanho_roi, inicio_roi
    )

    assert imagem_final.GetSize() == (
        RESOLUCAO, RESOLUCAO, profundidade
    )
    assert mascara_final.GetSize() == imagem_final.GetSize()

    img_final = sitk.GetArrayFromImage(imagem_final)
    lb_final = sitk.GetArrayFromImage(mascara_final)

    assert np.isfinite(img_final).all()
    assert np.isin(lb_final, [0, 1, 2, 3, 4]).all()

    # 4. Preservar volumes
    destino_img = DESTINO / f"image_{pid}.nii.gz"
    destino_lb = DESTINO / f"label_{pid}.nii.gz"

    estado_img = salvar_seguro(imagem_final, destino_img)
    estado_lb = salvar_seguro(mascara_final, destino_lb)

    total_fatias += profundidade

    manifesto.append({
        "paciente": pid,
        "fatias_originais": int(imagem.GetSize()[2]),
        "fatias_finais": int(profundidade),
        "shape_zyx": list(img_final.shape),
        "spacing_xyz": list(imagem_final.GetSpacing()),
        "percentil_99_5": limite,
        "intensidade_minima": float(img_final.min()),
        "intensidade_maxima": float(img_final.max()),
        "classes_presentes": [
            int(x) for x in np.unique(lb_final)
        ],
        "pixels_por_classe": {
            str(c): int(np.sum(lb_final == c))
            for c in range(1, 5)
        },
        "recorte_inicio_xy": [inicio_x, inicio_y],
        "imagem": estado_img,
        "mascara": estado_lb
    })

    print(
        f"{pid:8d} | "
        f"{imagem.GetSize()[2]:10d} | "
        f"{profundidade:7d} | "
        f"{RESOLUCAO}x{RESOLUCAO:3d}  | "
        f"{img_final.max():.1f}"
    )

# 5. Registrar parâmetros e resultados
config = {
    "dataset": "CHAOS T2-SPIR",
    "origem": str(ORIGEM),
    "destino": str(DESTINO),
    "percentil_superior": PERCENTIL,
    "spacing_alvo_xyz": list(SPACING_ALVO),
    "resolucao_xy": RESOLUCAO,
    "interpolacao_imagem": "linear",
    "interpolacao_mascara": "linear por classe + np.rint",
    "diferenca_original": (
        "Tamanho reamostrado calculado pela extensao fisica; "
        "evita voxel adicional artificial de int(sz+1)."
    ),
    "normalizacao_0_255": False,
    "pacientes": manifesto
}

manifesto_path = DESTINO / "manifesto_preprocessamento.json"

conteudo = json.dumps(
    config, indent=2, ensure_ascii=False
)

if manifesto_path.exists():
    assert manifesto_path.read_text(encoding="utf-8") == conteudo
else:
    temporario = manifesto_path.with_suffix(".json.tmp")
    temporario.write_text(conteudo, encoding="utf-8")
    os.replace(temporario, manifesto_path)

print("\n=== RESUMO FINAL ===")
print("Pacientes processados:", len(manifesto))
print("Fatias após reamostragem:", total_fatias)
print("Volumes de imagem:", len(list(DESTINO.glob("image_*.nii.gz"))))
print("Volumes de máscara:", len(list(DESTINO.glob("label_*.nii.gz"))))
print("Spacing:", SPACING_ALVO)
print("Resolução:", (256, 256))
print("Manifesto:", manifesto_path)

In [ ]:
from pathlib import Path
from collections import Counter
import SimpleITK as sitk
import numpy as np
import json
import os

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

DADOS = REPO / "data/CHAOST2/chaos_MR_T2_normalized"

CLASSES = {
    0: "BG",
    1: "LIVER",
    2: "RK",
    3: "LK",
    4: "SPLEEN",
}

classmap = {
    nome: {str(pid): [] for pid in range(20)}
    for nome in CLASSES.values()
}

pixels_por_classe = Counter()
fatias_com_classe = Counter()
resumo_pacientes = []

print("=== CLASSMAP CHAOS-T2 ===")
print("Paciente | Fatias | Fígado | Rim D | Rim E | Baço")
print("-" * 60)

for pid in range(20):
    img_path = DADOS / f"image_{pid}.nii.gz"
    lb_path = DADOS / f"label_{pid}.nii.gz"

    imagem = sitk.ReadImage(str(img_path))
    mascara = sitk.ReadImage(str(lb_path))

    assert imagem.GetSize() == mascara.GetSize()
    assert np.allclose(imagem.GetSpacing(), mascara.GetSpacing())
    assert np.allclose(imagem.GetOrigin(), mascara.GetOrigin())
    assert np.allclose(imagem.GetDirection(), mascara.GetDirection())

    img = sitk.GetArrayFromImage(imagem)
    lb = sitk.GetArrayFromImage(mascara)

    assert img.shape == lb.shape
    assert img.shape[1:] == (256, 256)
    assert np.isfinite(img).all()
    assert np.isin(lb, list(CLASSES)).all()

    contagem = Counter()

    for z in range(lb.shape[0]):
        valores, quantidades = np.unique(
            lb[z], return_counts=True
        )

        for valor, quantidade in zip(valores, quantidades):
            classe = int(valor)
            quantidade = int(quantidade)

            pixels_por_classe[classe] += quantidade

            if quantidade >= 1:
                classmap[CLASSES[classe]][str(pid)].append(z)
                fatias_com_classe[classe] += 1
                contagem[classe] += 1

    resumo_pacientes.append({
        "paciente": pid,
        "fatias": int(lb.shape[0]),
        "fatias_por_classe": {
            CLASSES[c]: contagem[c]
            for c in range(1, 5)
        },
        "intensidade_min": float(img.min()),
        "intensidade_max": float(img.max()),
    })

    print(
        f"{pid:8d} | "
        f"{lb.shape[0]:6d} | "
        f"{contagem[1]:6d} | "
        f"{contagem[2]:5d} | "
        f"{contagem[3]:5d} | "
        f"{contagem[4]:4d}"
    )

assert len(resumo_pacientes) == 20
assert sum(r["fatias"] for r in resumo_pacientes) == 713

# Verificar que todas as estruturas existem no dataset.
for classe in range(1, 5):
    assert fatias_com_classe[classe] > 0, (
        f"Nenhuma fatia encontrada para {CLASSES[classe]}"
    )

# Salvar classmap esperado pelo pipeline.
caminho_classmap = DADOS / "classmap_1.json"

conteudo = json.dumps(
    classmap,
    indent=2,
    ensure_ascii=False
)

if caminho_classmap.exists():
    assert json.loads(
        caminho_classmap.read_text(encoding="utf-8")
    ) == classmap, "Classmap existente difere do calculado."
else:
    temporario = DADOS / "classmap_1.json.tmp"
    temporario.write_text(conteudo, encoding="utf-8")
    os.replace(temporario, caminho_classmap)

print("\n=== RESUMO GLOBAL ===")

for classe, nome in CLASSES.items():
    pacientes_positivos = sum(
        bool(classmap[nome][str(pid)])
        for pid in range(20)
    )

    print(
        f"{nome:8s} | "
        f"{fatias_com_classe[classe]:4d} fatias | "
        f"{pacientes_positivos:2d} pacientes | "
        f"{pixels_por_classe[classe]:,} pixels"
    )

print("\n=== INTENSIDADES MRI ===")

print(
    "Menor intensidade:",
    min(r["intensidade_min"] for r in resumo_pacientes)
)
print(
    "Maior intensidade:",
    max(r["intensidade_max"] for r in resumo_pacientes)
)

print("\nClassmap salvo:", caminho_classmap)
print("Verificação concluída.")

In [ ]:
from pathlib import Path
import subprocess
import sys
import json
import os

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
EXP = REPO / "exp/eft_mri_chaos_v1"
DADOS = REPO / "data/CHAOST2/chaos_MR_T2_normalized"

EXP.mkdir(parents=True, exist_ok=True)

SCRIPT = EXP / "gerar_superpixels_mri.py"
LOG = EXP / "superpixels.log"
STATUS = EXP / "status_superpixels.json"
LANCAMENTO = EXP / "lancamento_superpixels.json"

assert all(
    (DADOS / f"image_{pid}.nii.gz").is_file()
    for pid in range(20)
)
assert (DADOS / "classmap_1.json").is_file()

codigo = r'''
from pathlib import Path
from scipy.ndimage import binary_fill_holes
from skimage.segmentation import felzenszwalb
from skimage.measure import label
import SimpleITK as sitk
import numpy as np
import json
import os
import traceback

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
EXP = REPO / "exp/eft_mri_chaos_v1"
DADOS = REPO / "data/CHAOST2/chaos_MR_T2_normalized"

STATUS = EXP / "status_superpixels.json"
MANIFESTO = EXP / "manifesto_superpixels.json"

LIMIAR = 50.0001
MIN_SIZE = 400
SIGMA = 1
MODE = "MIDDLE"


def salvar_json(caminho, dados):
    tmp = caminho.with_name(caminho.name + ".tmp")
    tmp.write_text(
        json.dumps(dados, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    os.replace(tmp, caminho)


def mascara_foreground(imagem):
    # Reprodução do fg_mask2d original.
    mascara = (imagem > LIMIAR).astype(np.float32)

    if mascara.max() < 0.999:
        return np.zeros(imagem.shape, dtype=np.uint8)

    componentes = label(mascara)

    contagens = np.bincount(componentes.ravel())[1:]
    maior = np.argmax(contagens) + 1

    mascara = componentes == maior
    mascara = binary_fill_holes(mascara)

    return mascara.astype(np.uint8)


def aplicar_mascara_original(superpixels, foreground):
    # Preserva o remapeamento do superpix_masking
    # presente no notebook original.
    regioes = np.int32(superpixels).copy()
    valores = np.unique(regioes)
    maior_valor = int(valores.max())

    regioes[regioes == 0] = maior_valor + 1

    valores = list(valores)
    valores.append(maior_valor)

    regioes = regioes * foreground.astype(np.int32)

    saida = np.zeros(regioes.shape, dtype=np.int32)
    proximo_rotulo = 1

    for valor in valores:
        if valor == 0:
            continue

        saida[regioes == valor] = proximo_rotulo
        proximo_rotulo += 1

    return saida


def salvar_nifti(array, referencia, destino):
    obj = sitk.GetImageFromArray(array)
    obj.CopyInformation(referencia)

    if destino.exists():
        anterior = sitk.ReadImage(str(destino))

        assert anterior.GetSize() == obj.GetSize()
        assert np.allclose(
            anterior.GetSpacing(), obj.GetSpacing()
        )
        assert np.allclose(
            anterior.GetOrigin(), obj.GetOrigin()
        )
        assert np.allclose(
            anterior.GetDirection(), obj.GetDirection()
        )
        assert np.array_equal(
            sitk.GetArrayFromImage(anterior), array
        )
        return "verificado"

    tmp = destino.with_name(
        destino.name.replace(".nii.gz", ".tmp.nii.gz")
    )

    sitk.WriteImage(obj, str(tmp), True)
    os.replace(tmp, destino)
    return "criado"


def executar():
    total_fatias = 0
    pacientes_concluidos = 0
    registros = []

    for pid in range(20):
        imagem_path = DADOS / f"image_{pid}.nii.gz"
        fg_path = DADOS / f"fgmask_{pid}.nii.gz"
        sp_path = DADOS / f"superpix-{MODE}_{pid}.nii.gz"

        imagem_obj = sitk.ReadImage(str(imagem_path))
        volume = sitk.GetArrayFromImage(imagem_obj)

        assert volume.ndim == 3
        assert volume.shape[1:] == (256, 256)
        assert np.isfinite(volume).all()

        z, h, w = volume.shape

        fg_volume = np.zeros((z, h, w), dtype=np.uint8)
        sp_volume = np.zeros((z, h, w), dtype=np.int32)

        regioes_total = 0
        fatias_com_foreground = 0

        for fatia in range(z):
            img2d = volume[fatia]

            # Parâmetros originais: scale padrão,
            # min_size=400, sigma=1.
            segmentos = felzenszwalb(
                img2d,
                min_size=MIN_SIZE,
                sigma=SIGMA
            )

            fg = mascara_foreground(img2d)
            sp = aplicar_mascara_original(segmentos, fg)

            assert np.all(sp[fg == 0] == 0)

            fg_volume[fatia] = fg
            sp_volume[fatia] = sp

            regioes_total += int(np.unique(sp[sp > 0]).size)

            if fg.any():
                fatias_com_foreground += 1

        assert sp_volume.shape == volume.shape
        assert fg_volume.shape == volume.shape

        estado_fg = salvar_nifti(
            fg_volume, imagem_obj, fg_path
        )
        estado_sp = salvar_nifti(
            sp_volume, imagem_obj, sp_path
        )

        total_fatias += z
        pacientes_concluidos += 1

        registro = {
            "paciente": pid,
            "fatias": z,
            "fatias_com_foreground": fatias_com_foreground,
            "regioes_total": regioes_total,
            "foreground_pixels": int(fg_volume.sum()),
            "arquivo_fgmask": fg_path.name,
            "arquivo_superpixels": sp_path.name,
            "estado_fg": estado_fg,
            "estado_superpixels": estado_sp
        }

        registros.append(registro)

        salvar_json(STATUS, {
            "estado": "executando",
            "pacientes_concluidos": pacientes_concluidos,
            "pacientes_total": 20,
            "fatias_processadas": total_fatias,
            "fatias_previstas": 713,
            "ultimo_paciente": pid,
            "ultimo_registro": registro
        })

        print(
            f"[{pacientes_concluidos:02d}/20] "
            f"Paciente {pid:02d} | "
            f"fatias={z} | "
            f"foreground={fatias_com_foreground} | "
            f"regioes={regioes_total}",
            flush=True
        )

    assert total_fatias == 713

    salvar_json(MANIFESTO, {
        "dataset": "CHAOS T2-SPIR",
        "metodo": "Felzenszwalb",
        "mode": MODE,
        "min_size": MIN_SIZE,
        "sigma": SIGMA,
        "scale": 1,
        "fg_thresh": LIMIAR,
        "mascara": "Maior componente conexo + preenchimento",
        "remapeamento": "superpix_masking original",
        "pacientes": registros
    })

    salvar_json(STATUS, {
        "estado": "concluido",
        "pacientes_concluidos": 20,
        "pacientes_total": 20,
        "fatias_processadas": total_fatias,
        "fatias_previstas": 713,
        "manifesto": str(MANIFESTO)
    })

    print("SUPERPIXELS MRI CONCLUÍDOS", flush=True)


if __name__ == "__main__":
    try:
        executar()
    except Exception:
        erro = traceback.format_exc()

        salvar_json(STATUS, {
            "estado": "erro",
            "mensagem": erro
        })

        print(erro, flush=True)
        raise
'''

SCRIPT.write_text(codigo, encoding="utf-8")

# Validar dependências antes de iniciar o processo.
verificacao = subprocess.run(
    [
        sys.executable,
        "-c",
        "import numpy, scipy, skimage, SimpleITK"
    ],
    capture_output=True,
    text=True
)

assert verificacao.returncode == 0, verificacao.stderr

# Evitar lançamento duplicado.
pid_anterior = None

if LANCAMENTO.exists():
    registro = json.loads(
        LANCAMENTO.read_text(encoding="utf-8")
    )
    pid_anterior = registro.get("pid")

ativo = False

if pid_anterior:
    try:
        cmdline = Path(
            f"/proc/{pid_anterior}/cmdline"
        ).read_bytes()
        ativo = b"gerar_superpixels_mri.py" in cmdline
    except OSError:
        pass

if ativo:
    print("Processo já está em execução.")
    print("PID:", pid_anterior)
else:
    with LOG.open("a", encoding="utf-8") as log:
        processo = subprocess.Popen(
            [sys.executable, "-u", str(SCRIPT)],
            cwd=str(REPO),
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True
        )

    registro = {
        "pid": processo.pid,
        "script": str(SCRIPT),
        "log": str(LOG),
        "status": str(STATUS),
        "modo": "detached"
    }

    LANCAMENTO.write_text(
        json.dumps(registro, indent=2),
        encoding="utf-8"
    )

    print("=== SUPERPIXELS CHAOS MRI ===")
    print("PID:", processo.pid)
    print("Método: Felzenszwalb")
    print("Configuração: MIDDLE / min_size=400 / sigma=1")
    print("Foreground: intensidade > 50.0001")
    print("Pacientes:", 20)
    print("Fatias:", 713)
    print("Modo: detached")
    print("Log:", LOG)
    print("Lançamento solicitado; conclusão ainda não verificada.")

In [ ]:
from pathlib import Path
import json
import os

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
EXP = REPO / "exp/eft_mri_chaos_v1"
DADOS = REPO / "data/CHAOST2/chaos_MR_T2_normalized"

LANCAMENTO = EXP / "lancamento_superpixels.json"
STATUS = EXP / "status_superpixels.json"
LOG = EXP / "superpixels.log"
MANIFESTO = EXP / "manifesto_superpixels.json"

def ler_json(caminho):
    if not caminho.is_file():
        return {}
    try:
        return json.loads(caminho.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return {}

registro = ler_json(LANCAMENTO)
status = ler_json(STATUS)

pid = registro.get("pid")
ativo = False

if pid:
    try:
        proc = Path(f"/proc/{pid}")
        cmdline = (proc / "cmdline").read_bytes()
        estado = (proc / "stat").read_text().split(") ")[1][0]

        ativo = (
            b"gerar_superpixels_mri.py" in cmdline
            and estado not in ("Z", "X")
        )
    except (OSError, IndexError):
        pass

fgmask = sorted(DADOS.glob("fgmask_*.nii.gz"))
superpixels = sorted(DADOS.glob("superpix-MIDDLE_*.nii.gz"))

concluidos = status.get("pacientes_concluidos", 0)
total = 20
progresso = min(100, concluidos / total * 100)

print("=" * 60)
print("EFT MRI — GERAÇÃO DE SUPERPIXELS")
print("=" * 60)

print("PID:", pid)
print("Processo ativo:", "SIM" if ativo else "NÃO")
print("Estado:", status.get("estado", "inicializando"))

print(f"\nPacientes: {concluidos}/{total}")
print(
    "Fatias processadas:",
    status.get("fatias_processadas", 0),
    "/ 713"
)

print(f"Progresso: {progresso:.1f}%")

blocos = int(progresso * 30 / 100)
print("[" + "█" * blocos + "░" * (30 - blocos) + "]")

print("\n=== ARQUIVOS GERADOS ===")
print("Foreground masks:", len(fgmask), "/ 20")
print("Volumes de superpixels:", len(superpixels), "/ 20")
print("Manifesto final:", "SIM" if MANIFESTO.is_file() else "NÃO")

print("\n=== DIAGNÓSTICO ===")

if status.get("estado") == "concluido":
    if (
        len(fgmask) == 20
        and len(superpixels) == 20
        and MANIFESTO.is_file()
    ):
        print("GERAÇÃO CONCLUÍDA — 20 PACIENTES")
    else:
        print("ATENÇÃO: conclusão registrada, mas faltam arquivos.")

elif status.get("estado") == "erro":
    print("ERRO DETECTADO")
    print(status.get("mensagem", "Erro não especificado"))

elif ativo:
    print("PROCESSAMENTO EM EXECUÇÃO")

else:
    print("Processo inativo sem conclusão confirmada.")

if status.get("ultimo_registro"):
    ultimo = status["ultimo_registro"]

    print("\n=== ÚLTIMO PACIENTE ===")
    print("Paciente:", ultimo.get("paciente"))
    print("Fatias:", ultimo.get("fatias"))
    print("Fatias com foreground:", ultimo.get("fatias_com_foreground"))
    print("Regiões de superpixels:", ultimo.get("regioes_total"))

print("\n=== ÚLTIMAS MENSAGENS DO LOG ===")

if LOG.is_file():
    with LOG.open(encoding="utf-8", errors="replace") as arquivo:
        linhas = arquivo.readlines()

    for linha in linhas[-15:]:
        print(linha.rstrip())
else:
    print("Log ainda não criado.")

In [ ]:
from pathlib import Path
import numpy as np
import SimpleITK as sitk
import json

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
DADOS = REPO / "data/CHAOST2/chaos_MR_T2_normalized"
EXP = REPO / "exp/eft_mri_chaos_v1"

manifesto = json.loads(
    (EXP / "manifesto_superpixels.json").read_text()
)

assert len(manifesto["pacientes"]) == 20

MIN_AREA = 400
MAX_AREA = 10000
EXCLUIR_CLASSES = (2, 3)  # Rins: configuração label_set=0

totais = {
    "fatias": 0,
    "fatias_elegiveis": 0,
    "regioes_candidatas": 0,
    "pixels_fg": 0,
    "pixels_cobertos": 0,
    "pixels_rins": 0,
    "pixels_rins_cobertos": 0,
}

print("=== 1. QUALIDADE DOS SUPERPIXELS ===")
print(
    "Paciente | Fatias | Elegíveis | Regiões | "
    "Cobertura FG | Cobertura rins"
)
print("-" * 82)

for pid in range(20):
    imagem = sitk.ReadImage(str(DADOS / f"image_{pid}.nii.gz"))
    fg = sitk.GetArrayFromImage(
        sitk.ReadImage(str(DADOS / f"fgmask_{pid}.nii.gz"))
    )
    sp = sitk.GetArrayFromImage(
        sitk.ReadImage(
            str(DADOS / f"superpix-MIDDLE_{pid}.nii.gz")
        )
    )
    lb = sitk.GetArrayFromImage(
        sitk.ReadImage(str(DADOS / f"label_{pid}.nii.gz"))
    )

    assert sp.shape == fg.shape == lb.shape
    assert sp.shape == sitk.GetArrayFromImage(imagem).shape
    assert np.all(sp[fg == 0] == 0)

    regioes = 0
    elegiveis = 0

    for z in range(sp.shape[0]):
        # Não utilizar classes de teste na seleção
        # de episódios EFT.
        if np.isin(lb[z], EXCLUIR_CLASSES).any():
            continue

        rotulos, contagens = np.unique(
            sp[z][sp[z] > 0],
            return_counts=True
        )

        validos = (
            (contagens >= MIN_AREA)
            & (contagens <= MAX_AREA)
        )

        n = int(validos.sum())
        regioes += n

        if n > 0:
            elegiveis += 1

    pixels_fg = int((fg > 0).sum())
    pixels_cobertos = int(((sp > 0) & (fg > 0)).sum())

    rins = np.isin(lb, EXCLUIR_CLASSES)
    pixels_rins = int(rins.sum())
    pixels_rins_cobertos = int((rins & (sp > 0)).sum())

    cobertura_fg = (
        100 * pixels_cobertos / pixels_fg
        if pixels_fg else 0
    )

    cobertura_rins = (
        100 * pixels_rins_cobertos / pixels_rins
        if pixels_rins else 0
    )

    print(
        f"{pid:8d} | "
        f"{sp.shape[0]:6d} | "
        f"{elegiveis:9d} | "
        f"{regioes:7d} | "
        f"{cobertura_fg:10.1f}% | "
        f"{cobertura_rins:12.1f}%"
    )

    totais["fatias"] += sp.shape[0]
    totais["fatias_elegiveis"] += elegiveis
    totais["regioes_candidatas"] += regioes
    totais["pixels_fg"] += pixels_fg
    totais["pixels_cobertos"] += pixels_cobertos
    totais["pixels_rins"] += pixels_rins
    totais["pixels_rins_cobertos"] += pixels_rins_cobertos

assert totais["fatias"] == 713

print("\n=== 2. RESUMO GLOBAL ===")
print("Fatias:", totais["fatias"])
print(
    "Fatias elegíveis antes da divisão de pacientes:",
    totais["fatias_elegiveis"]
)
print(
    "Regiões candidatas antes da divisão de pacientes:",
    totais["regioes_candidatas"]
)

for nome, numerador, denominador in [
    ("Cobertura FG",
     totais["pixels_cobertos"], totais["pixels_fg"]),
    ("Cobertura dos rins",
     totais["pixels_rins_cobertos"], totais["pixels_rins"])
]:
    pct = 100 * numerador / denominador if denominador else 0
    print(f"{nome}: {pct:.2f}%")

print("\n=== 3. DIVISÃO DOS PACIENTES NO REPOSITÓRIO ===")

arquivo = REPO / "dataloaders/GenericSuperDatasetv2.py"

if arquivo.exists():
    linhas = arquivo.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()

    termos = (
        "idx_split", "pid_curr_load", "scan_ids",
        "fold", "pid_split", "scan_per_load"
    )

    encontrados = [
        i for i, linha in enumerate(linhas)
        if any(t in linha.lower() for t in termos)
    ]

    for i in encontrados[:18]:
        inicio = max(0, i - 2)
        fim = min(len(linhas), i + 4)

        print(f"\n--- Linhas {inicio+1}-{fim} ---")
        for j in range(inicio, fim):
            print(f"{j+1:04d}: {linhas[j]}")
else:
    print("Dataloader não encontrado.")

print("\n=== 4. SCRIPTS CT REUTILIZÁVEIS ===")

for nome in [
    "exp/eft_ct_piloto200_v1/treinar_ct.py",
    "exp/eft_ct_piloto200_v1/continuar_10000.py"
]:
    caminho = REPO / nome
    print(
        caminho.relative_to(REPO),
        "ENCONTRADO" if caminho.is_file() else "AUSENTE"
    )

print("\nAuditoria concluída. Nenhum arquivo modificado.")

In [ ]:
from pathlib import Path
from collections import Counter
import SimpleITK as sitk
import numpy as np
import json
import os
import re

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
DADOS = REPO / "data/CHAOST2/chaos_MR_T2_normalized"
EXP = REPO / "exp/eft_mri_chaos_v1"

QUERIES = list(range(0, 4))
SUPORTE = [4]
TREINO = list(range(5, 20))

assert sorted(QUERIES + SUPORTE + TREINO) == list(range(20))

MIN_AREA = 400
MAX_AREA = 10000
CLASSES_EXCLUIDAS = [2, 3]

episodios = []
resumo = []

print("=== 1. DIVISÃO DO FOLD 0 ===")
print("Queries:", QUERIES)
print("Suporte:", SUPORTE)
print("Treinamento:", TREINO)
print("Classes excluídas:", CLASSES_EXCLUIDAS)

print("\n=== 2. EPISÓDIOS ELEGÍVEIS PARA EFT ===")
print("Paciente | Fatias válidas | Regiões candidatas")
print("-" * 52)

for pid in TREINO:
    sp = sitk.GetArrayFromImage(
        sitk.ReadImage(
            str(DADOS / f"superpix-MIDDLE_{pid}.nii.gz")
        )
    )

    lb = sitk.GetArrayFromImage(
        sitk.ReadImage(
            str(DADOS / f"label_{pid}.nii.gz")
        )
    )

    assert sp.shape == lb.shape

    fatias_validas = 0
    regioes_validas = 0

    for z in range(sp.shape[0]):
        # Excluir fatias contendo classes de avaliação.
        if np.isin(lb[z], CLASSES_EXCLUIDAS).any():
            continue

        rotulos, contagens = np.unique(
            sp[z][sp[z] > 0],
            return_counts=True
        )

        for rotulo, area in zip(rotulos, contagens):
            if MIN_AREA <= area <= MAX_AREA:
                episodios.append({
                    "paciente": int(pid),
                    "fatia": int(z),
                    "superpixel": int(rotulo),
                    "area": int(area)
                })
                regioes_validas += 1

        if any(
            ep["paciente"] == pid and ep["fatia"] == z
            for ep in episodios[-regioes_validas:]
        ) if regioes_validas else False:
            fatias_validas += 1

    resumo.append({
        "paciente": pid,
        "fatias_validas": fatias_validas,
        "regioes_validas": regioes_validas
    })

    print(
        f"{pid:8d} | "
        f"{fatias_validas:14d} | "
        f"{regioes_validas:18d}"
    )

assert episodios, "Nenhum episódio de treinamento encontrado."

# Verificação contra vazamento de pacientes.
assert not (
    set(ep["paciente"] for ep in episodios)
    & set(QUERIES + SUPORTE)
)

# Verificação contra vazamento de classes de avaliação.
for pid, z in sorted(set(
    (ep["paciente"], ep["fatia"])
    for ep in episodios
)):
    pass  # A exclusão já foi aplicada acima.

plano = {
    "dataset": "CHAOST2_Superpix",
    "fold": 0,
    "label_set": 0,
    "resolucao": 256,
    "queries": QUERIES,
    "suporte": SUPORTE,
    "treino": TREINO,
    "classes_excluidas": CLASSES_EXCLUIDAS,
    "min_area": MIN_AREA,
    "max_area": MAX_AREA,
    "total_regioes": len(episodios),
    "resumo": resumo,
    "episodios": episodios
}

EXP.mkdir(parents=True, exist_ok=True)
CAMINHO_PLANO = EXP / "plano_eft_mri_fold0.json"

conteudo = json.dumps(
    plano, indent=2, ensure_ascii=False
)

if CAMINHO_PLANO.exists():
    anterior = json.loads(
        CAMINHO_PLANO.read_text(encoding="utf-8")
    )
    assert anterior == plano, "Plano existente difere do calculado."
else:
    tmp = CAMINHO_PLANO.with_suffix(".json.tmp")
    tmp.write_text(conteudo, encoding="utf-8")
    os.replace(tmp, CAMINHO_PLANO)

print("\n=== 3. RESUMO ===")
print("Pacientes de treinamento:", len(TREINO))
print("Fatias elegíveis:", sum(r["fatias_validas"] for r in resumo))
print("Regiões candidatas:", len(episodios))
print("Plano:", CAMINHO_PLANO)

print("\n=== 4. ESTRUTURA DOS SCRIPTS CT ===")

for relativo in [
    "exp/eft_ct_piloto200_v1/treinar_ct.py",
    "exp/eft_ct_piloto200_v1/continuar_10000.py"
]:
    arquivo = REPO / relativo

    print(f"\n--- {relativo} ---")

    if not arquivo.exists():
        print("Arquivo ausente")
        continue

    linhas = arquivo.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()

    print("Total de linhas:", len(linhas))

    for numero, linha in enumerate(linhas, 1):
        texto = linha.strip()

        if (
            texto.startswith("def ")
            or texto.startswith("class ")
            or re.search(
                r"checkpoint|learning_rate|"
                r"weight_decay|AdamW|"
                r"modelo|optimizer|"
                r"n_steps|total_steps",
                texto,
                re.IGNORECASE
            )
        ):
            if len(texto) > 150:
                texto = texto[:150] + "..."
            print(f"{numero:04d}: {texto}")

print("\nPreparação concluída.")

In [ ]:
from pathlib import Path

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

SCRIPT = (
    REPO
    / "exp/eft_ct_piloto200_v1/treinar_ct.py"
)

assert SCRIPT.is_file()

linhas = SCRIPT.read_text(
    encoding="utf-8",
    errors="replace"
).splitlines()

def mostrar(titulo, inicio, fim):
    print(f"\n{'=' * 65}")
    print(titulo)
    print(f"{'=' * 65}")

    for i in range(inicio - 1, min(fim, len(linhas))):
        print(f"{i + 1:04d}: {linhas[i]}")

mostrar(
    "1. IMPORTS, CONFIGURAÇÃO E CAMINHOS",
    1, 56
)

mostrar(
    "2. CARREGAMENTO DOS VOLUMES E EPISÓDIOS",
    57, 115
)

mostrar(
    "3. MODELO, OTIMIZADOR E LOOP DE TREINAMENTO",
    135, 265
)

mostrar(
    "4. ATUALIZAÇÃO, CHECKPOINTS E FINALIZAÇÃO",
    266, 319
)

print("\nInspeção concluída. Nenhum arquivo modificado.")

In [ ]:
from pathlib import Path
import importlib.util
import random
import gc
import json

import numpy as np
import torch
import torch.nn.functional as F
import SimpleITK as sitk

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
DADOS = ROOT / "data/CHAOST2/chaos_MR_T2_normalized"
EXP = ROOT / "exp/eft_mri_chaos_v1"
PLANO = EXP / "plano_eft_mri_fold0.json"
SCRIPT_CT = ROOT / "exp/eft_ct_piloto200_v1/treinar_ct.py"

assert PLANO.is_file()
assert SCRIPT_CT.is_file()
assert torch.cuda.is_available()

# Reutilizar exatamente a construção do modelo e o
# tratamento de alinhamento já validados no CT.
spec = importlib.util.spec_from_file_location(
    "eft_ct_reuso_diagnostico", SCRIPT_CT
)
base = importlib.util.module_from_spec(spec)
spec.loader.exec_module(base)

plano = json.loads(PLANO.read_text(encoding="utf-8"))
assert plano["fold"] == 0
assert plano["classes_excluidas"] == [2, 3]
assert plano["resolucao"] == 256
assert len(plano["episodios"]) == 5996

SEMENTE = 1234
random.seed(SEMENTE)
np.random.seed(SEMENTE)
torch.manual_seed(SEMENTE)
torch.cuda.manual_seed_all(SEMENTE)

torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False

# Seleção determinística de uma região candidata.
ep = random.Random(SEMENTE).choice(plano["episodios"])

pid = int(ep["paciente"])
z = int(ep["fatia"])
rotulo = int(ep["superpixel"])

assert pid in plano["treino"]
assert pid not in plano["queries"] + plano["suporte"]

volume = sitk.GetArrayFromImage(
    sitk.ReadImage(str(DADOS / f"image_{pid}.nii.gz"))
).astype(np.float32)

superpixels = sitk.GetArrayFromImage(
    sitk.ReadImage(str(DADOS / f"superpix-MIDDLE_{pid}.nii.gz"))
)

anatomia = sitk.GetArrayFromImage(
    sitk.ReadImage(str(DADOS / f"label_{pid}.nii.gz"))
)

assert volume.shape == superpixels.shape == anatomia.shape
assert not np.isin(anatomia[z], [2, 3]).any()

mascara = (superpixels[z] == rotulo)
assert int(mascara.sum()) == int(ep["area"])

# Normalização MRI original: z-score por volume.
media = float(volume.mean())
desvio = float(volume.std())

assert np.isfinite(media)
assert np.isfinite(desvio) and desvio > 0

volume_normalizado = (volume - media) / desvio
fatia = volume_normalizado[z]

rgb = np.repeat(fatia[..., None], 3, axis=2)

def tensor_imagem(imagem):
    return torch.from_numpy(
        np.ascontiguousarray(imagem)
    ).permute(2, 0, 1)[None].to(
        "cuda", dtype=torch.float32
    )

support = tensor_imagem(rgb)
query = tensor_imagem(np.fliplr(rgb))

fg = torch.from_numpy(
    mascara.copy()
)[None].to("cuda", dtype=torch.float32)

alvo = torch.from_numpy(
    np.fliplr(mascara).copy()
)[None].to("cuda", dtype=torch.long)

print("=== 1. ENTRADA MRI ===")
print("GPU:", torch.cuda.get_device_name(0))
print("Paciente:", pid)
print("Fatia:", z)
print("Superpixel:", rotulo)
print("Área da região:", int(mascara.sum()))
print("Média do volume original:", round(media, 4))
print("Desvio do volume original:", round(desvio, 4))
print(
    "Média / desvio após normalização:",
    round(float(volume_normalizado.mean()), 5),
    round(float(volume_normalizado.std()), 5)
)
print("Support:", tuple(support.shape))
print("Query:", tuple(query.shape))
print(
    "Intervalo da entrada:",
    round(float(support.min()), 4),
    round(float(support.max()), 4)
)

assert support.shape == query.shape == (1, 3, 256, 256)
assert fg.shape == alvo.shape == (1, 256, 256)
assert torch.isfinite(support).all().item()

print("\n=== 2. CONSTRUÇÃO DO VMAMBA-TINY ===")

modelo = base.construir_modelo("tiny", repo=ROOT)
estado_align = base.instalar(modelo)

modelo.to("cuda")
modelo.train()

otimizador = torch.optim.AdamW(
    modelo.parameters(),
    lr=1e-5,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=0.01
)

parametro = next(
    p for p in modelo.encoder.parameters()
    if p.requires_grad
)
peso_antes = parametro.detach().clone()

print("Modelo construído.")
print("Otimizador: AdamW")
print("Learning rate: 1e-5")
print("Weight decay: 0.01")

print("\n=== 3. FORWARD E LOSS ===")

otimizador.zero_grad(set_to_none=True)
estado_align["ultimo"] = None

retorno = modelo(
    [[support]],
    [[fg]],
    [[1 - fg]],
    [query],
    isval=False,
    val_wsize=2,
    show_viz=False
)

logits, alinhamento = retorno[:2]

assert logits.shape == (1, 2, 256, 256)

ce = F.cross_entropy(logits, alvo)
loss = ce + alinhamento

assert torch.isfinite(logits).all().item()
assert torch.isfinite(loss).item()

print("Logits:", tuple(logits.shape))
print("CE:", round(float(ce.detach()), 6))
print(
    "Alinhamento:",
    round(float(
        alinhamento.detach()
        if torch.is_tensor(alinhamento)
        else alinhamento
    ), 6)
)
print("Loss total:", round(float(loss.detach()), 6))

print("\n=== 4. BACKWARD ===")

loss.backward()

gradientes = [
    p.grad
    for p in modelo.encoder.parameters()
    if p.grad is not None
]

gradientes_nao_nulos = sum(
    torch.count_nonzero(g).item() > 0
    for g in gradientes
)

assert gradientes
assert gradientes_nao_nulos > 0
assert all(
    torch.isfinite(g).all().item()
    for g in gradientes
)

print("Tensores com gradiente:", len(gradientes))
print("Gradientes não nulos:", gradientes_nao_nulos)
print("Gradientes finitos: SIM")

print("\n=== 5. ATUALIZAÇÃO DOS PESOS ===")

otimizador.step()
torch.cuda.synchronize()

maior_alteracao = float(
    (parametro.detach() - peso_antes).abs().max()
)

assert maior_alteracao > 0, (
    "Não foi detectada atualização no parâmetro monitorado."
)

print("Maior alteração no parâmetro:", maior_alteracao)
print("Atualização dos pesos: CONFIRMADA")
print("Exceção de alinhamento:", estado_align["ultimo"])

print("\nDIAGNÓSTICO EFT-MRI CONCLUÍDO")

# Descartar o modelo de diagnóstico:
# nenhum checkpoint é criado.
del modelo, otimizador, retorno
del logits, alinhamento, ce, loss
del gradientes, parametro, peso_antes
del support, query, fg, alvo

gc.collect()
torch.cuda.empty_cache()

In [ ]:
from pathlib import Path
import subprocess
import hashlib
import random
import json
import sys
import os
import ast

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

EXP = ROOT / "exp/eft_mri_chaos_v1"
RUN = ROOT / "reproducao/eft_mri_chaost2_fold0_piloto200"

SCRIPT_CT = ROOT / "exp/eft_ct_piloto200_v1/treinar_ct.py"
SCRIPT_MRI = EXP / "treinar_mri_200.py"

PLANO_ORIGEM = EXP / "plano_eft_mri_fold0.json"
PLANO_200 = RUN / "plano_200.json"
PLANO_10000 = RUN / "plano_10000.json"

LOG = RUN / "console.log"
LANCAMENTO = RUN / "lancamento.json"

SEMENTE = 1234
PASSOS_PILOTO = 200
PASSOS_TOTAIS = 10000

assert SCRIPT_CT.is_file()
assert PLANO_ORIGEM.is_file()

EXP.mkdir(parents=True, exist_ok=True)
RUN.mkdir(parents=True, exist_ok=True)


def escrever_json_protegido(caminho, objeto):
    if caminho.exists():
        anterior = json.loads(
            caminho.read_text(encoding="utf-8")
        )
        assert anterior == objeto, (
            f"Arquivo existente diferente: {caminho}"
        )
        return

    temp = caminho.with_name(caminho.name + ".tmp")
    temp.write_text(
        json.dumps(objeto, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    os.replace(temp, caminho)


def sha256(caminho):
    h = hashlib.sha256()
    with caminho.open("rb") as f:
        for bloco in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(bloco)
    return h.hexdigest()


# ============================================================
# 1. PREPARAR A SEQUÊNCIA REPRODUZÍVEL DE EPISÓDIOS
# ============================================================

plano_base = json.loads(
    PLANO_ORIGEM.read_text(encoding="utf-8")
)

assert plano_base["fold"] == 0
assert plano_base["classes_excluidas"] == [2, 3]
assert plano_base["resolucao"] == 256

candidatos = plano_base["episodios"]
assert len(candidatos) == 5996

rng = random.Random(SEMENTE)

sequencia = []

for passo in range(1, PASSOS_TOTAIS + 1):
    candidato = candidatos[rng.randrange(len(candidatos))]

    sequencia.append({
        "passo": passo,
        "paciente": int(candidato["paciente"]),
        "fatia": int(candidato["fatia"]),
        "regiao": int(candidato["superpixel"]),
        "area": int(candidato["area"])
    })

assert all(
    ep["paciente"] in range(5, 20)
    for ep in sequencia
)

config_comum = {
    "dataset": "CHAOST2",
    "modalidade": "MRI T2-SPIR",
    "fold": 0,
    "label_set": 0,
    "resolucao": 256,
    "classes_excluidas": [2, 3],
    "treino": list(range(5, 20)),
    "queries": [0, 1, 2, 3],
    "suporte": [4],
    "semente": SEMENTE,
    "normalizacao": "z-score por volume MRI",
    "amostragem": "uniforme com reposicao entre regioes candidatas",
    "total_candidatos": len(candidatos),
}

escrever_json_protegido(
    PLANO_10000,
    {
        **config_comum,
        "passos": PASSOS_TOTAIS,
        "episodios": sequencia
    }
)

escrever_json_protegido(
    PLANO_200,
    {
        **config_comum,
        "passos": PASSOS_PILOTO,
        "episodios": sequencia[:PASSOS_PILOTO]
    }
)

assert (
    json.loads(PLANO_10000.read_text())["episodios"][:200]
    == json.loads(PLANO_200.read_text())["episodios"]
)

print("=== 1. PLANEJAMENTO ===")
print("Regiões candidatas:", len(candidatos))
print("Episódios do piloto:", PASSOS_PILOTO)
print("Episódios da sequência completa:", PASSOS_TOTAIS)
print("Semente:", SEMENTE)
print("Plano piloto:", PLANO_200)
print("Plano completo:", PLANO_10000)


# ============================================================
# 2. ADAPTAR O SCRIPT CT SEM MODIFICAR O ORIGINAL
# ============================================================

codigo = SCRIPT_CT.read_text(encoding="utf-8")


def substituir_exato(antigo, novo):
    global codigo

    quantidade = codigo.count(antigo)
    assert quantidade == 1, (
        f"Substituição incompatível: {antigo!r}; "
        f"ocorrências={quantidade}"
    )
    codigo = codigo.replace(antigo, novo, 1)


substituir_exato(
    'DADOS = ROOT / "data/SABS/sabs_CT_normalized"',
    'DADOS = ROOT / "data/CHAOST2/chaos_MR_T2_normalized"'
)

substituir_exato(
    'RUN = ROOT / "reproducao/eft_ct_felzenszwalb_fold0_piloto200"',
    'RUN = ROOT / "reproducao/eft_mri_chaost2_fold0_piloto200"'
)

substituir_exato(
    '    return imagem, regioes',
    '''    # MRI: normalização pela média e desvio do volume.
    imagem = imagem.astype(np.float32)
    media_volume = float(imagem.mean())
    desvio_volume = float(imagem.std())

    assert np.isfinite(media_volume)
    assert np.isfinite(desvio_volume)
    assert desvio_volume > 1e-6

    imagem = (imagem - media_volume) / desvio_volume

    return imagem, regioes'''
)

substituir_exato(
    '    return (t - media) / desvio',
    '    return t  # Entrada MRI ja normalizada por volume'
)

substituir_exato(
    'assert pid in range(7, 30)',
    'assert pid in range(5, 20)'
)

substituir_exato(
    'classmap["KID_R"]',
    'classmap["RK"]'
)

substituir_exato(
    'classmap["KID_l"]',
    'classmap["LK"]'
)

substituir_exato(
    '"dataset": "SABS"',
    '"dataset": "CHAOST2"'
)

substituir_exato(
    '"modalidade": "CT"',
    '"modalidade": "MRI T2-SPIR"'
)

# Garantir que a adaptação conserva o treinamento
# original e contém o tratamento MRI esperado.
assert 'modelo = construir_modelo("tiny", repo=ROOT)' in codigo
assert 'loss = ce + alinhamento' in codigo
assert 'lr=LR' in codigo
assert 'weight_decay=WD' in codigo
assert 'imagem = (imagem - media_volume) / desvio_volume' in codigo

# Conferência sintática sem executar o treinamento.
ast.parse(codigo, filename=str(SCRIPT_MRI))

if SCRIPT_MRI.exists():
    assert (
        SCRIPT_MRI.read_text(encoding="utf-8") == codigo
    ), "O script MRI existente difere da adaptação."
else:
    SCRIPT_MRI.write_text(codigo, encoding="utf-8")

print("\n=== 2. ADAPTAÇÃO ===")
print("Script original CT preservado:", SCRIPT_CT.is_file())
print("Script MRI:", SCRIPT_MRI)
print("SHA256 script MRI:", sha256(SCRIPT_MRI))
print("Loss: CE + alinhamento")
print("Otimizador: AdamW")
print("Learning rate: 1e-5")
print("Weight decay: 0.01")
print("Normalização: z-score por volume")


# ============================================================
# 3. IMPEDIR EXECUÇÃO DUPLICADA
# ============================================================

if (RUN / "checkpoint_0000.pt").exists():
    print("\nATENÇÃO: já existe checkpoint inicial.")
    print("Nenhum novo processo foi iniciado.")
else:

    ativo = False

    if LANCAMENTO.exists():
        anterior = json.loads(
            LANCAMENTO.read_text(encoding="utf-8")
        )
        pid_anterior = anterior.get("pid")

        if pid_anterior:
            try:
                cmd = Path(
                    f"/proc/{pid_anterior}/cmdline"
                ).read_bytes()

                stat = Path(
                    f"/proc/{pid_anterior}/stat"
                ).read_text().split(") ")[1][0]

                ativo = (
                    b"treinar_mri_200.py" in cmd
                    and stat not in ("Z", "X")
                )
            except (OSError, IndexError):
                pass

    if ativo:
        print("\nTreinamento já está em execução.")
        print("PID:", pid_anterior)

    else:
        with LOG.open("a", encoding="utf-8") as log:
            processo = subprocess.Popen(
                [
                    sys.executable,
                    "-u",
                    str(SCRIPT_MRI)
                ],
                cwd=str(ROOT),
                stdin=subprocess.DEVNULL,
                stdout=log,
                stderr=subprocess.STDOUT,
                start_new_session=True
            )

        escrever_json_protegido(
            LANCAMENTO,
            {
                "pid": processo.pid,
                "script": str(SCRIPT_MRI),
                "log": str(LOG),
                "modo": "detached",
                "passos": PASSOS_PILOTO
            }
        )

        print("\n=== 3. TREINAMENTO LANÇADO ===")
        print("PID:", processo.pid)
        print("Experimento: EFT-MRI CHAOS-T2")
        print("Encoder: VMamba-Tiny")
        print("Passos:", PASSOS_PILOTO)
        print("Checkpoints previstos: 0, 50, 100 e 200")
        print("Log:", LOG)
        print("Modo: detached")
        print("Lançamento solicitado; execução ainda não verificada.")

In [ ]:
from pathlib import Path
import json
import statistics

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
RUN = ROOT / "reproducao/eft_mri_chaost2_fold0_piloto200"

LANCAMENTO = RUN / "lancamento.json"
STATUS = RUN / "status.json"
PASSOS = RUN / "passos.jsonl"
LOG = RUN / "console.log"

def ler_json(caminho):
    if not caminho.is_file():
        return {}
    try:
        return json.loads(caminho.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return {}

lancamento = ler_json(LANCAMENTO)
status = ler_json(STATUS)

pid = lancamento.get("pid")
ativo = False

if pid:
    try:
        proc = Path(f"/proc/{pid}")
        cmdline = (proc / "cmdline").read_bytes()
        estado_proc = (proc / "stat").read_text().split(") ")[1][0]

        ativo = (
            b"treinar_mri_200.py" in cmdline
            and estado_proc not in ("Z", "X")
        )
    except (OSError, IndexError):
        pass

historico = []

if PASSOS.is_file():
    with PASSOS.open(encoding="utf-8") as arquivo:
        for linha in arquivo:
            try:
                historico.append(json.loads(linha))
            except json.JSONDecodeError:
                pass

passo = max(
    int(status.get("passo", 0)),
    max((int(p["passo"]) for p in historico), default=0)
)

total = 200
progresso = min(100, passo / total * 100)

print("=" * 60)
print("EFT-MRI CHAOS-T2 — VMAMBA-TINY")
print("=" * 60)

print("PID:", pid)
print("Processo ativo:", "SIM" if ativo else "NÃO")
print("Estado:", status.get("estado", "inicializando"))
print(f"Passo: {passo}/{total}")
print(f"Progresso: {progresso:.1f}%")

blocos = int(progresso * 30 / 100)
print("[" + "█" * blocos + "░" * (30 - blocos) + "]")

print("\n=== CHECKPOINTS ===")

for cp in [0, 50, 100, 200]:
    caminho = RUN / f"checkpoint_{cp:04d}.pt"
    print(
        f"Passo {cp:3d}:",
        "SALVO" if caminho.is_file() else "pendente"
    )

print("\n=== EVOLUÇÃO DAS PERDAS ===")

if historico:
    print(
        f"{'Intervalo':<16}"
        f"{'CE':>10}"
        f"{'Alinh.':>12}"
        f"{'Loss':>12}"
    )
    print("-" * 50)

    for n in [20, 50]:
        trecho = historico[-n:]

        if not trecho:
            continue

        inicio = trecho[0]["passo"]
        fim = trecho[-1]["passo"]

        ce = statistics.mean(
            float(p["ce"]) for p in trecho
        )
        align = statistics.mean(
            float(p["alinhamento"]) for p in trecho
        )
        loss = statistics.mean(
            float(p["loss"]) for p in trecho
        )

        print(
            f"{inicio:4d}–{fim:<10d}"
            f"{ce:10.4f}"
            f"{align:12.4f}"
            f"{loss:12.4f}"
        )

    ultimo = historico[-1]

    print("\nÚltimo passo registrado:")
    print("Passo:", ultimo["passo"])
    print("CE:", round(float(ultimo["ce"]), 6))
    print(
        "Alinhamento:",
        round(float(ultimo["alinhamento"]), 6)
    )
    print("Loss:", round(float(ultimo["loss"]), 6))
    print(
        "Tempo por passo:",
        round(float(ultimo["segundos"]), 3),
        "s"
    )
else:
    print("Nenhum passo registrado ainda.")

print("\n=== DIAGNÓSTICO ===")

if status.get("estado") == "erro":
    print("ERRO NO TREINAMENTO")
    print(status.get("traceback", "Sem traceback registrado."))

elif status.get("estado") == "concluido":
    if (RUN / "checkpoint_0200.pt").is_file():
        print("PILOTO CONCLUÍDO — CHECKPOINT 200 CONFIRMADO")
    else:
        print("ATENÇÃO: treinamento concluído sem checkpoint 200.")

elif ativo:
    print("TREINAMENTO EM EXECUÇÃO")

else:
    print("Processo inativo sem conclusão confirmada.")

print("\n=== ÚLTIMAS MENSAGENS ===")

if LOG.is_file():
    with LOG.open(
        encoding="utf-8", errors="replace"
    ) as arquivo:
        linhas = arquivo.readlines()

    for linha in linhas[-15:]:
        print(linha.rstrip())
else:
    print("Log ainda não encontrado.")

In [ ]:
from pathlib import Path
import subprocess
import hashlib
import json
import ast
import sys
import os

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
EXP = ROOT / "exp/eft_mri_chaos_v1"

PILOTO = ROOT / "reproducao/eft_mri_chaost2_fold0_piloto200"
RUN = ROOT / "reproducao/eft_mri_chaost2_fold0_10000_v1"

SCRIPT = EXP / "continuar_mri_10000.py"
ORIGEM = PILOTO / "checkpoint_0200.pt"
PLANO = PILOTO / "plano_10000.json"
BASE = EXP / "treinar_mri_200.py"

LOG = RUN / "console.log"
LANCAMENTO = RUN / "lancamento.json"

assert ORIGEM.is_file(), "Checkpoint 200 ausente"
assert PLANO.is_file(), "Plano 10.000 ausente"
assert BASE.is_file(), "Script MRI piloto ausente"

plano = json.loads(PLANO.read_text(encoding="utf-8"))
assert len(plano["episodios"]) == 10000
assert plano["episodios"][199]["passo"] == 200
assert plano["episodios"][-1]["passo"] == 10000

RUN.mkdir(parents=True, exist_ok=True)

codigo = r'''
import os
import sys
import time
import json
import random
import hashlib
import shutil
import traceback
import importlib.util
import fcntl

from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
EXP = ROOT / "exp/eft_mri_chaos_v1"
PILOTO = ROOT / "reproducao/eft_mri_chaost2_fold0_piloto200"
RUN = ROOT / "reproducao/eft_mri_chaost2_fold0_10000_v1"

BASE = EXP / "treinar_mri_200.py"
PLANO = PILOTO / "plano_10000.json"
ORIGEM = PILOTO / "checkpoint_0200.pt"

LATEST = RUN / "checkpoint_latest.pt"
STATUS = RUN / "status.json"
HISTORICO = RUN / "passos.jsonl"
CONFIG = RUN / "config_continuacao.json"

TOTAL = 10000
INICIAL = 200
INTERVALO_RECUPERACAO = 50
INTERVALO_HISTORICO = 500

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))


def sha256(caminho):
    h = hashlib.sha256()
    with caminho.open("rb") as f:
        for bloco in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(bloco)
    return h.hexdigest()


def json_atomico(caminho, dados):
    tmp = caminho.with_name(caminho.name + ".tmp")
    tmp.write_text(
        json.dumps(dados, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    os.replace(tmp, caminho)


def preservar_checkpoint(passo):
    destino = RUN / f"checkpoint_{passo:05d}.pt"

    if destino.exists():
        return

    tmp = destino.with_name(destino.name + ".tmp")
    shutil.copyfile(LATEST, tmp)
    os.replace(tmp, destino)

    print(
        f"Checkpoint histórico preservado: {passo}",
        flush=True
    )


def salvar_checkpoint(passo, modelo, otimizador, procedencia):
    estado = {
        "passo": passo,
        "modelo": modelo.state_dict(),
        "otimizador": otimizador.state_dict(),
        "rng_python": random.getstate(),
        "rng_numpy": np.random.get_state(),
        "rng_torch": torch.get_rng_state(),
        "rng_cuda": torch.cuda.get_rng_state_all(),
        "procedencia": procedencia
    }

    tmp = LATEST.with_name(LATEST.name + ".tmp")
    torch.save(estado, tmp)
    os.replace(tmp, LATEST)

    if passo % INTERVALO_HISTORICO == 0:
        preservar_checkpoint(passo)

    print(f"Checkpoint de recuperação: {passo}", flush=True)


def ajustar_historico(passo_restaurado):
    # Se houve interrupção entre dois checkpoints,
    # descartar registros posteriores ao estado
    # efetivamente restaurado.
    if not HISTORICO.exists():
        assert passo_restaurado == INICIAL
        return

    linhas_validas = []
    ultimo = INICIAL

    for linha in HISTORICO.read_text(
        encoding="utf-8"
    ).splitlines():
        try:
            registro = json.loads(linha)
        except json.JSONDecodeError:
            continue

        passo = int(registro["passo"])

        if passo > passo_restaurado:
            break

        assert passo == ultimo + 1, (
            f"Histórico descontínuo: {ultimo} -> {passo}"
        )

        linhas_validas.append(
            json.dumps(registro, ensure_ascii=False)
        )
        ultimo = passo

    assert ultimo == passo_restaurado, (
        f"Histórico termina em {ultimo}, "
        f"checkpoint está no passo {passo_restaurado}"
    )

    tmp = HISTORICO.with_name(HISTORICO.name + ".tmp")
    tmp.write_text(
        "\n".join(linhas_validas)
        + ("\n" if linhas_validas else ""),
        encoding="utf-8"
    )
    os.replace(tmp, HISTORICO)


def executar():
    assert torch.cuda.is_available()
    assert ORIGEM.is_file()
    assert BASE.is_file()

    plano = json.loads(
        PLANO.read_text(encoding="utf-8")
    )

    episodios = plano["episodios"]

    assert len(episodios) == TOTAL
    assert plano["fold"] == 0
    assert plano["classes_excluidas"] == [2, 3]
    assert plano["resolucao"] == 256

    for i, ep in enumerate(episodios):
        assert int(ep["passo"]) == i + 1
        assert int(ep["paciente"]) in range(5, 20)
        assert int(ep["area"]) >= 400

    procedencia = {
        "checkpoint_origem": str(ORIGEM),
        "checkpoint_origem_sha256": sha256(ORIGEM),
        "plano_sha256": sha256(PLANO),
        "script_piloto_sha256": sha256(BASE)
    }

    config = {
        "dataset": "CHAOST2",
        "modalidade": "MRI T2-SPIR",
        "encoder": "VMamba-Tiny",
        "fold": 0,
        "passo_inicial": INICIAL,
        "passo_final": TOTAL,
        "normalizacao": "z-score por volume",
        "loss": "CE + alinhamento",
        "otimizador": "AdamW",
        "lr": 1e-5,
        "weight_decay": 0.01,
        "checkpoint_recuperacao": INTERVALO_RECUPERACAO,
        "checkpoint_historico": INTERVALO_HISTORICO,
        "procedencia": procedencia
    }

    if CONFIG.exists():
        assert json.loads(
            CONFIG.read_text(encoding="utf-8")
        ) == config, "Configuração de retomada diferente."
    else:
        json_atomico(CONFIG, config)

    spec = importlib.util.spec_from_file_location(
        "eft_mri_base_continuacao", BASE
    )
    base = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(base)

    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False

    print("Construindo VMamba-Tiny...", flush=True)

    modelo = base.construir_modelo("tiny", repo=ROOT)
    estado_align = base.instalar(modelo)
    modelo.to("cuda")

    otimizador = torch.optim.AdamW(
        modelo.parameters(),
        lr=1e-5,
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=0.01
    )

    # Retomar do checkpoint mais recente, quando existir.
    fonte = LATEST if LATEST.exists() else ORIGEM

    checkpoint = torch.load(
        fonte,
        map_location="cpu",
        weights_only=False
    )

    passo_inicial = int(checkpoint["passo"])

    assert INICIAL <= passo_inicial <= TOTAL

    if fonte == LATEST:
        assert checkpoint["procedencia"] == procedencia
    else:
        assert passo_inicial == INICIAL

    modelo.load_state_dict(
        checkpoint["modelo"], strict=True
    )

    otimizador.load_state_dict(
        checkpoint["otimizador"]
    )

    # Reconstituir os geradores aleatórios do
    # checkpoint, sem reiniciar a semente.
    random.setstate(checkpoint["rng_python"])
    np.random.set_state(checkpoint["rng_numpy"])
    torch.set_rng_state(checkpoint["rng_torch"])
    torch.cuda.set_rng_state_all(
        checkpoint["rng_cuda"]
    )

    del checkpoint

    modelo.train()

    ajustar_historico(passo_inicial)

    if (
        passo_inicial % INTERVALO_HISTORICO == 0
        and fonte == LATEST
    ):
        preservar_checkpoint(passo_inicial)

    print(
        f"Checkpoint restaurado: passo {passo_inicial}",
        flush=True
    )
    print(
        f"Próximo passo: {passo_inicial + 1}",
        flush=True
    )
    print("Meta: 10000 passos", flush=True)

    inicio_total = time.time()

    json_atomico(STATUS, {
        "estado": "executando",
        "passo": passo_inicial,
        "total": TOTAL,
        "checkpoint_origem": str(fonte)
    })

    with HISTORICO.open(
        "a", encoding="utf-8", buffering=1
    ) as log:

        for passo in range(passo_inicial + 1, TOTAL + 1):
            inicio = time.time()
            ep = episodios[passo - 1]

            support, fg, query, alvo = base.episodio(ep)

            otimizador.zero_grad(set_to_none=True)
            estado_align["ultimo"] = None

            retorno = modelo(
                [[support]],
                [[fg]],
                [[1 - fg]],
                [query],
                isval=False,
                val_wsize=2,
                show_viz=False
            )

            logits, alinhamento = retorno[:2]

            assert logits.shape == (1, 2, 256, 256)

            ce = F.cross_entropy(logits, alvo)
            loss = ce + alinhamento

            assert torch.isfinite(loss).item()

            loss.backward()

            grads = [
                p.grad
                for p in modelo.encoder.parameters()
                if p.grad is not None
            ]

            assert grads
            assert any(
                torch.count_nonzero(g).item() > 0
                for g in grads
            )
            assert all(
                torch.isfinite(g).all().item()
                for g in grads
            )

            otimizador.step()
            torch.cuda.synchronize()

            registro = {
                "passo": passo,
                "paciente": ep["paciente"],
                "fatia": ep["fatia"],
                "regiao": ep["regiao"],
                "ce": float(ce.detach()),
                "alinhamento": float(
                    alinhamento.detach()
                    if torch.is_tensor(alinhamento)
                    else alinhamento
                ),
                "loss": float(loss.detach()),
                "segundos": time.time() - inicio,
                "alinhamento_excecao": estado_align["ultimo"]
            }

            log.write(json.dumps(registro) + "\n")

            if passo % 10 == 0:
                print(
                    f"Passo {passo}/{TOTAL} | "
                    f"CE={registro['ce']:.4f} | "
                    f"Align={registro['alinhamento']:.4f} | "
                    f"Loss={registro['loss']:.4f}",
                    flush=True
                )

            if passo % INTERVALO_RECUPERACAO == 0:
                salvar_checkpoint(
                    passo, modelo, otimizador, procedencia
                )

            if passo % 10 == 0 or passo == TOTAL:
                json_atomico(STATUS, {
                    "estado": "executando",
                    "passo": passo,
                    "total": TOTAL,
                    "loss": registro["loss"],
                    "ce": registro["ce"],
                    "alinhamento": registro["alinhamento"],
                    "tempo_decorrido_s": time.time() - inicio_total
                })

    assert LATEST.exists()

    json_atomico(STATUS, {
        "estado": "concluido",
        "passo": TOTAL,
        "total": TOTAL,
        "tempo_decorrido_s": time.time() - inicio_total
    })

    print("EFT-MRI 10000 PASSOS CONCLUÍDO", flush=True)


if __name__ == "__main__":
    # Bloqueio para impedir dois processos
    # de treinamento na mesma execução.
    RUN.mkdir(parents=True, exist_ok=True)

    with (RUN / "execucao.lock").open("w") as lock:
        try:
            fcntl.flock(
                lock.fileno(),
                fcntl.LOCK_EX | fcntl.LOCK_NB
            )
        except BlockingIOError:
            raise RuntimeError(
                "Outra execução EFT-MRI já está ativa."
            )

        try:
            executar()
        except Exception:
            erro = traceback.format_exc()

            json_atomico(STATUS, {
                "estado": "erro",
                "traceback": erro
            })

            print(erro, flush=True)
            raise
'''

# Validar sintaxe antes de salvar.
ast.parse(codigo, filename=str(SCRIPT))

if SCRIPT.exists():
    assert SCRIPT.read_text(encoding="utf-8") == codigo, (
        "O script de continuação existente é diferente."
    )
else:
    SCRIPT.write_text(codigo, encoding="utf-8")

def ler_json(caminho):
    if not caminho.exists():
        return {}
    return json.loads(caminho.read_text(encoding="utf-8"))

status = ler_json(RUN / "status.json")

# Conferir se a execução anterior permanece ativa.
ativo = False
pid_anterior = None

if LANCAMENTO.exists():
    registro = ler_json(LANCAMENTO)
    pid_anterior = registro.get("pid")

    if pid_anterior:
        try:
            proc = Path(f"/proc/{pid_anterior}")
            cmd = (proc / "cmdline").read_bytes()
            estado = (proc / "stat").read_text().split(") ")[1][0]

            ativo = (
                b"continuar_mri_10000.py" in cmd
                and estado not in ("Z", "X")
            )
        except (OSError, IndexError):
            pass

print("=== CONTINUAÇÃO EFT-MRI ===")
print("Origem: checkpoint 200 do piloto")
print("Destino: passo 10000")
print("Plano:", PLANO)
print("Script:", SCRIPT)
print("Checkpoints de recuperação: a cada 50 passos")
print("Checkpoints históricos: a cada 500 passos")

if status.get("estado") == "concluido":
    print("\nTreinamento já concluído. Nenhum novo processo iniciado.")

elif ativo:
    print("\nTreinamento já está ativo.")
    print("PID:", pid_anterior)

else:
    with LOG.open("a", encoding="utf-8") as log:
        processo = subprocess.Popen(
            [sys.executable, "-u", str(SCRIPT)],
            cwd=str(ROOT),
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True
        )

    lancamento = {
        "pid": processo.pid,
        "script": str(SCRIPT),
        "log": str(LOG),
        "passo_origem": 200,
        "passo_final": 10000,
        "modo": "detached"
    }

    temporario = LANCAMENTO.with_suffix(".json.tmp")
    temporario.write_text(
        json.dumps(lancamento, indent=2),
        encoding="utf-8"
    )
    os.replace(temporario, LANCAMENTO)

    print("\nTREINAMENTO LANÇADO")
    print("PID:", processo.pid)
    print("Log:", LOG)
    print("Diretório dos checkpoints:", RUN)
    print("Lançamento solicitado; execução ainda não verificada.")

In [ ]:
from pathlib import Path
import subprocess
import hashlib
import json
import ast
import sys
import os

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
EXP = ROOT / "exp/eft_mri_chaos_v1"

PILOTO = ROOT / "reproducao/eft_mri_chaost2_fold0_piloto200"
RUN = ROOT / "reproducao/eft_mri_chaost2_fold0_10000_v1"

SCRIPT = EXP / "continuar_mri_10000.py"
ORIGEM = PILOTO / "checkpoint_0200.pt"
PLANO = PILOTO / "plano_10000.json"
BASE = EXP / "treinar_mri_200.py"

LOG = RUN / "console.log"
LANCAMENTO = RUN / "lancamento.json"

assert ORIGEM.is_file(), "Checkpoint 200 ausente"
assert PLANO.is_file(), "Plano 10.000 ausente"
assert BASE.is_file(), "Script MRI piloto ausente"

plano = json.loads(PLANO.read_text(encoding="utf-8"))
assert len(plano["episodios"]) == 10000
assert plano["episodios"][199]["passo"] == 200
assert plano["episodios"][-1]["passo"] == 10000

RUN.mkdir(parents=True, exist_ok=True)

codigo = r'''
import os
import sys
import time
import json
import random
import hashlib
import shutil
import traceback
import importlib.util
import fcntl

from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
EXP = ROOT / "exp/eft_mri_chaos_v1"
PILOTO = ROOT / "reproducao/eft_mri_chaost2_fold0_piloto200"
RUN = ROOT / "reproducao/eft_mri_chaost2_fold0_10000_v1"

BASE = EXP / "treinar_mri_200.py"
PLANO = PILOTO / "plano_10000.json"
ORIGEM = PILOTO / "checkpoint_0200.pt"

LATEST = RUN / "checkpoint_latest.pt"
STATUS = RUN / "status.json"
HISTORICO = RUN / "passos.jsonl"
CONFIG = RUN / "config_continuacao.json"

TOTAL = 10000
INICIAL = 200
INTERVALO_RECUPERACAO = 50
INTERVALO_HISTORICO = 500

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))


def sha256(caminho):
    h = hashlib.sha256()
    with caminho.open("rb") as f:
        for bloco in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(bloco)
    return h.hexdigest()


def json_atomico(caminho, dados):
    tmp = caminho.with_name(caminho.name + ".tmp")
    tmp.write_text(
        json.dumps(dados, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    os.replace(tmp, caminho)


def preservar_checkpoint(passo):
    destino = RUN / f"checkpoint_{passo:05d}.pt"

    if destino.exists():
        return

    tmp = destino.with_name(destino.name + ".tmp")
    shutil.copyfile(LATEST, tmp)
    os.replace(tmp, destino)

    print(
        f"Checkpoint histórico preservado: {passo}",
        flush=True
    )


def salvar_checkpoint(passo, modelo, otimizador, procedencia):
    estado = {
        "passo": passo,
        "modelo": modelo.state_dict(),
        "otimizador": otimizador.state_dict(),
        "rng_python": random.getstate(),
        "rng_numpy": np.random.get_state(),
        "rng_torch": torch.get_rng_state(),
        "rng_cuda": torch.cuda.get_rng_state_all(),
        "procedencia": procedencia
    }

    tmp = LATEST.with_name(LATEST.name + ".tmp")
    torch.save(estado, tmp)
    os.replace(tmp, LATEST)

    if passo % INTERVALO_HISTORICO == 0:
        preservar_checkpoint(passo)

    print(f"Checkpoint de recuperação: {passo}", flush=True)


def ajustar_historico(passo_restaurado):
    # Se houve interrupção entre dois checkpoints,
    # descartar registros posteriores ao estado
    # efetivamente restaurado.
    if not HISTORICO.exists():
        assert passo_restaurado == INICIAL
        return

    linhas_validas = []
    ultimo = INICIAL

    for linha in HISTORICO.read_text(
        encoding="utf-8"
    ).splitlines():
        try:
            registro = json.loads(linha)
        except json.JSONDecodeError:
            continue

        passo = int(registro["passo"])

        if passo > passo_restaurado:
            break

        assert passo == ultimo + 1, (
            f"Histórico descontínuo: {ultimo} -> {passo}"
        )

        linhas_validas.append(
            json.dumps(registro, ensure_ascii=False)
        )
        ultimo = passo

    assert ultimo == passo_restaurado, (
        f"Histórico termina em {ultimo}, "
        f"checkpoint está no passo {passo_restaurado}"
    )

    tmp = HISTORICO.with_name(HISTORICO.name + ".tmp")
    tmp.write_text(
        "\n".join(linhas_validas)
        + ("\n" if linhas_validas else ""),
        encoding="utf-8"
    )
    os.replace(tmp, HISTORICO)


def executar():
    assert torch.cuda.is_available()
    assert ORIGEM.is_file()
    assert BASE.is_file()

    plano = json.loads(
        PLANO.read_text(encoding="utf-8")
    )

    episodios = plano["episodios"]

    assert len(episodios) == TOTAL
    assert plano["fold"] == 0
    assert plano["classes_excluidas"] == [2, 3]
    assert plano["resolucao"] == 256

    for i, ep in enumerate(episodios):
        assert int(ep["passo"]) == i + 1
        assert int(ep["paciente"]) in range(5, 20)
        assert int(ep["area"]) >= 400

    procedencia = {
        "checkpoint_origem": str(ORIGEM),
        "checkpoint_origem_sha256": sha256(ORIGEM),
        "plano_sha256": sha256(PLANO),
        "script_piloto_sha256": sha256(BASE)
    }

    config = {
        "dataset": "CHAOST2",
        "modalidade": "MRI T2-SPIR",
        "encoder": "VMamba-Tiny",
        "fold": 0,
        "passo_inicial": INICIAL,
        "passo_final": TOTAL,
        "normalizacao": "z-score por volume",
        "loss": "CE + alinhamento",
        "otimizador": "AdamW",
        "lr": 1e-5,
        "weight_decay": 0.01,
        "checkpoint_recuperacao": INTERVALO_RECUPERACAO,
        "checkpoint_historico": INTERVALO_HISTORICO,
        "procedencia": procedencia
    }

    if CONFIG.exists():
        assert json.loads(
            CONFIG.read_text(encoding="utf-8")
        ) == config, "Configuração de retomada diferente."
    else:
        json_atomico(CONFIG, config)

    spec = importlib.util.spec_from_file_location(
        "eft_mri_base_continuacao", BASE
    )
    base = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(base)

    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False

    print("Construindo VMamba-Tiny...", flush=True)

    modelo = base.construir_modelo("tiny", repo=ROOT)
    estado_align = base.instalar(modelo)
    modelo.to("cuda")

    otimizador = torch.optim.AdamW(
        modelo.parameters(),
        lr=1e-5,
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=0.01
    )

    # Retomar do checkpoint mais recente, quando existir.
    fonte = LATEST if LATEST.exists() else ORIGEM

    checkpoint = torch.load(
        fonte,
        map_location="cpu",
        weights_only=False
    )

    passo_inicial = int(checkpoint["passo"])

    assert INICIAL <= passo_inicial <= TOTAL

    if fonte == LATEST:
        assert checkpoint["procedencia"] == procedencia
    else:
        assert passo_inicial == INICIAL

    modelo.load_state_dict(
        checkpoint["modelo"], strict=True
    )

    otimizador.load_state_dict(
        checkpoint["otimizador"]
    )

    # Reconstituir os geradores aleatórios do
    # checkpoint, sem reiniciar a semente.
    random.setstate(checkpoint["rng_python"])
    np.random.set_state(checkpoint["rng_numpy"])
    torch.set_rng_state(checkpoint["rng_torch"])
    torch.cuda.set_rng_state_all(
        checkpoint["rng_cuda"]
    )

    del checkpoint

    modelo.train()

    ajustar_historico(passo_inicial)

    if (
        passo_inicial % INTERVALO_HISTORICO == 0
        and fonte == LATEST
    ):
        preservar_checkpoint(passo_inicial)

    print(
        f"Checkpoint restaurado: passo {passo_inicial}",
        flush=True
    )
    print(
        f"Próximo passo: {passo_inicial + 1}",
        flush=True
    )
    print("Meta: 10000 passos", flush=True)

    inicio_total = time.time()

    json_atomico(STATUS, {
        "estado": "executando",
        "passo": passo_inicial,
        "total": TOTAL,
        "checkpoint_origem": str(fonte)
    })

    with HISTORICO.open(
        "a", encoding="utf-8", buffering=1
    ) as log:

        for passo in range(passo_inicial + 1, TOTAL + 1):
            inicio = time.time()
            ep = episodios[passo - 1]

            support, fg, query, alvo = base.episodio(ep)

            otimizador.zero_grad(set_to_none=True)
            estado_align["ultimo"] = None

            retorno = modelo(
                [[support]],
                [[fg]],
                [[1 - fg]],
                [query],
                isval=False,
                val_wsize=2,
                show_viz=False
            )

            logits, alinhamento = retorno[:2]

            assert logits.shape == (1, 2, 256, 256)

            ce = F.cross_entropy(logits, alvo)
            loss = ce + alinhamento

            assert torch.isfinite(loss).item()

            loss.backward()

            grads = [
                p.grad
                for p in modelo.encoder.parameters()
                if p.grad is not None
            ]

            assert grads
            assert any(
                torch.count_nonzero(g).item() > 0
                for g in grads
            )
            assert all(
                torch.isfinite(g).all().item()
                for g in grads
            )

            otimizador.step()
            torch.cuda.synchronize()

            registro = {
                "passo": passo,
                "paciente": ep["paciente"],
                "fatia": ep["fatia"],
                "regiao": ep["regiao"],
                "ce": float(ce.detach()),
                "alinhamento": float(
                    alinhamento.detach()
                    if torch.is_tensor(alinhamento)
                    else alinhamento
                ),
                "loss": float(loss.detach()),
                "segundos": time.time() - inicio,
                "alinhamento_excecao": estado_align["ultimo"]
            }

            log.write(json.dumps(registro) + "\n")

            if passo % 10 == 0:
                print(
                    f"Passo {passo}/{TOTAL} | "
                    f"CE={registro['ce']:.4f} | "
                    f"Align={registro['alinhamento']:.4f} | "
                    f"Loss={registro['loss']:.4f}",
                    flush=True
                )

            if passo % INTERVALO_RECUPERACAO == 0:
                salvar_checkpoint(
                    passo, modelo, otimizador, procedencia
                )

            if passo % 10 == 0 or passo == TOTAL:
                json_atomico(STATUS, {
                    "estado": "executando",
                    "passo": passo,
                    "total": TOTAL,
                    "loss": registro["loss"],
                    "ce": registro["ce"],
                    "alinhamento": registro["alinhamento"],
                    "tempo_decorrido_s": time.time() - inicio_total
                })

    assert LATEST.exists()

    json_atomico(STATUS, {
        "estado": "concluido",
        "passo": TOTAL,
        "total": TOTAL,
        "tempo_decorrido_s": time.time() - inicio_total
    })

    print("EFT-MRI 10000 PASSOS CONCLUÍDO", flush=True)


if __name__ == "__main__":
    # Bloqueio para impedir dois processos
    # de treinamento na mesma execução.
    RUN.mkdir(parents=True, exist_ok=True)

    with (RUN / "execucao.lock").open("w") as lock:
        try:
            fcntl.flock(
                lock.fileno(),
                fcntl.LOCK_EX | fcntl.LOCK_NB
            )
        except BlockingIOError:
            raise RuntimeError(
                "Outra execução EFT-MRI já está ativa."
            )

        try:
            executar()
        except Exception:
            erro = traceback.format_exc()

            json_atomico(STATUS, {
                "estado": "erro",
                "traceback": erro
            })

            print(erro, flush=True)
            raise
'''

# Validar sintaxe antes de salvar.
ast.parse(codigo, filename=str(SCRIPT))

if SCRIPT.exists():
    assert SCRIPT.read_text(encoding="utf-8") == codigo, (
        "O script de continuação existente é diferente."
    )
else:
    SCRIPT.write_text(codigo, encoding="utf-8")

def ler_json(caminho):
    if not caminho.exists():
        return {}
    return json.loads(caminho.read_text(encoding="utf-8"))

status = ler_json(RUN / "status.json")

# Conferir se a execução anterior permanece ativa.
ativo = False
pid_anterior = None

if LANCAMENTO.exists():
    registro = ler_json(LANCAMENTO)
    pid_anterior = registro.get("pid")

    if pid_anterior:
        try:
            proc = Path(f"/proc/{pid_anterior}")
            cmd = (proc / "cmdline").read_bytes()
            estado = (proc / "stat").read_text().split(") ")[1][0]

            ativo = (
                b"continuar_mri_10000.py" in cmd
                and estado not in ("Z", "X")
            )
        except (OSError, IndexError):
            pass

print("=== CONTINUAÇÃO EFT-MRI ===")
print("Origem: checkpoint 200 do piloto")
print("Destino: passo 10000")
print("Plano:", PLANO)
print("Script:", SCRIPT)
print("Checkpoints de recuperação: a cada 50 passos")
print("Checkpoints históricos: a cada 500 passos")

if status.get("estado") == "concluido":
    print("\nTreinamento já concluído. Nenhum novo processo iniciado.")

elif ativo:
    print("\nTreinamento já está ativo.")
    print("PID:", pid_anterior)

else:
    with LOG.open("a", encoding="utf-8") as log:
        processo = subprocess.Popen(
            [sys.executable, "-u", str(SCRIPT)],
            cwd=str(ROOT),
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True
        )

    lancamento = {
        "pid": processo.pid,
        "script": str(SCRIPT),
        "log": str(LOG),
        "passo_origem": 200,
        "passo_final": 10000,
        "modo": "detached"
    }

    temporario = LANCAMENTO.with_suffix(".json.tmp")
    temporario.write_text(
        json.dumps(lancamento, indent=2),
        encoding="utf-8"
    )
    os.replace(temporario, LANCAMENTO)

    print("\nTREINAMENTO LANÇADO")
    print("PID:", processo.pid)
    print("Log:", LOG)
    print("Diretório dos checkpoints:", RUN)
    print("Lançamento solicitado; execução ainda não verificada.")

In [ ]:
from pathlib import Path
from collections import deque
import json
import statistics
import shutil
import re

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
RUN = ROOT / "reproducao/eft_mri_chaost2_fold0_10000_v1"

LANCAMENTO = RUN / "lancamento.json"
STATUS = RUN / "status.json"
PASSOS = RUN / "passos.jsonl"
LOG = RUN / "console.log"
LATEST = RUN / "checkpoint_latest.pt"

def ler_json(caminho):
    if not caminho.is_file():
        return {}
    try:
        return json.loads(caminho.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return {}

lancamento = ler_json(LANCAMENTO)
status = ler_json(STATUS)

# 1. Processo
pid = lancamento.get("pid")
ativo = False

if pid:
    try:
        proc = Path(f"/proc/{pid}")
        cmd = (proc / "cmdline").read_bytes()
        estado_proc = (
            (proc / "stat").read_text().split(") ")[1][0]
        )

        ativo = (
            b"continuar_mri_10000.py" in cmd
            and estado_proc not in ("Z", "X")
        )
    except (OSError, IndexError):
        pass

# 2. Histórico de treinamento
historico = []

if PASSOS.is_file():
    with PASSOS.open(encoding="utf-8") as f:
        for linha in f:
            try:
                historico.append(json.loads(linha))
            except json.JSONDecodeError:
                pass

passo_historico = max(
    (int(x["passo"]) for x in historico),
    default=200
)

passo_status = int(status.get("passo", 200))
passo = max(passo_historico, passo_status)

TOTAL = 10000
progresso = min(100, passo / TOTAL * 100)

print("=" * 64)
print("EFT-MRI CHAOS-T2 — VMAMBA-TINY")
print("=" * 64)

print("PID:", pid)
print("Processo ativo:", "SIM" if ativo else "NÃO")
print("Estado:", status.get("estado", "inicializando"))
print(f"Passo: {passo:,}/{TOTAL:,}")
print(f"Progresso: {progresso:.2f}%")

blocos = int(progresso * 30 / 100)
print("[" + "█" * blocos + "░" * (30 - blocos) + "]")

# 3. Checkpoints
print("\n=== CHECKPOINTS ===")

print(
    "Checkpoint original (200):",
    "SIM" if (
        ROOT
        / "reproducao/eft_mri_chaost2_fold0_piloto200/"
        "checkpoint_0200.pt"
    ).is_file() else "NÃO"
)

print(
    "Checkpoint de recuperação:",
    "SIM" if LATEST.is_file() else "ainda não criado"
)

historicos = sorted(
    (
        int(m.group(1))
        for p in RUN.glob("checkpoint_*.pt")
        if (m := re.fullmatch(
            r"checkpoint_(\d{5})\.pt", p.name
        ))
    )
)

print("Checkpoints históricos preservados:", historicos)

if LATEST.is_file():
    print(
        "Tamanho do checkpoint latest:",
        round(LATEST.stat().st_size / 1024**2, 1),
        "MiB"
    )

# 4. Perdas
print("\n=== EVOLUÇÃO DA LOSS ===")

if historico:
    print(
        f"{'Intervalo':<18}"
        f"{'CE':>10}"
        f"{'Alinh.':>12}"
        f"{'Loss':>12}"
    )
    print("-" * 52)

    for quantidade in [20, 100, 500]:
        trecho = historico[-quantidade:]

        ce = statistics.mean(float(x["ce"]) for x in trecho)
        alinh = statistics.mean(
            float(x["alinhamento"]) for x in trecho
        )
        loss = statistics.mean(float(x["loss"]) for x in trecho)

        print(
            f"{trecho[0]['passo']:5d}–"
            f"{trecho[-1]['passo']:<11d}"
            f"{ce:10.4f}"
            f"{alinh:12.4f}"
            f"{loss:12.4f}"
        )

    # Médias por intervalos completos de 500 passos
    print("\n=== ÚLTIMOS BLOCOS DE 500 PASSOS ===")

    blocos_500 = {}

    for registro in historico:
        p = int(registro["passo"])
        indice = (p - 1) // 500
        blocos_500.setdefault(indice, []).append(
            float(registro["loss"])
        )

    for indice in sorted(blocos_500)[-6:]:
        valores = blocos_500[indice]
        inicio = indice * 500 + 1
        fim = indice * 500 + len(valores)

        print(
            f"Passos {inicio:5d}–{fim:<5d} | "
            f"Loss média: {statistics.mean(valores):.4f} "
            f"| n={len(valores)}"
        )

    ultimo = historico[-1]

    print("\n=== ÚLTIMO PASSO ===")
    print("Passo:", ultimo["passo"])
    print("CE:", round(float(ultimo["ce"]), 6))
    print(
        "Alinhamento:",
        round(float(ultimo["alinhamento"]), 6)
    )
    print("Loss:", round(float(ultimo["loss"]), 6))
    print(
        "Tempo do passo:",
        round(float(ultimo["segundos"]), 3),
        "s"
    )

else:
    print("Aguardando o primeiro passo após o checkpoint 200.")

# 5. Espaço disponível
livre_gib = shutil.disk_usage(RUN).free / 1024**3

print("\n=== ARMAZENAMENTO ===")
print(f"Espaço livre: {livre_gib:.2f} GiB")

if livre_gib < 5:
    print("ATENÇÃO: pouco espaço para os próximos checkpoints.")

# 6. Diagnóstico
print("\n=== DIAGNÓSTICO ===")

if status.get("estado") == "erro":
    print("ERRO NO TREINAMENTO")
    print(status.get("traceback", "Sem detalhes."))

elif status.get("estado") == "concluido":
    if passo == TOTAL and LATEST.is_file():
        print("TREINAMENTO CONCLUÍDO — 10.000 PASSOS")
    else:
        print("ATENÇÃO: conclusão ainda não confirmada integralmente.")

elif ativo:
    print("TREINAMENTO EM EXECUÇÃO")

else:
    print("Processo inativo sem conclusão confirmada.")

# 7. Log recente
print("\n=== ÚLTIMAS MENSAGENS DO LOG ===")

if LOG.is_file():
    with LOG.open(
        encoding="utf-8", errors="replace"
    ) as f:
        ultimas = deque(f, maxlen=15)

    for linha in ultimas:
        print(linha.rstrip())
else:
    print("Log ainda não encontrado.")

In [ ]:
from pathlib import Path
import os
import re

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

BASES = [
    ROOT / "exp",
    ROOT / "reproducao",
]

ALVOS = [
    "avaliacao_alp_rins_positivas_f0_7ckpts",
    "avaliacao_alp_sam_rins_f0_4ckpts",
]

resultados = []

ignorados = {
    ".git", "__pycache__", "logs", "snapshots",
    "source", "site-packages", ".ipynb_checkpoints"
}

for base in BASES:
    if not base.exists():
        continue

    for diretorio, subpastas, arquivos in os.walk(base):
        subpastas[:] = [
            d for d in subpastas
            if d not in ignorados
        ]

        for nome in arquivos:
            if not nome.endswith((".py", ".ipynb")):
                continue

            caminho = Path(diretorio) / nome

            if caminho.stat().st_size > 5_000_000:
                continue

            texto = caminho.read_text(
                encoding="utf-8", errors="replace"
            )

            pontos = 0

            if any(alvo in texto for alvo in ALVOS):
                pontos += 100

            if "eft_ct_felzenszwalb_fold0" in texto:
                pontos += 30

            if "KID_R" in texto and "KID_l" in texto:
                pontos += 15

            if "checkpoint" in texto.lower():
                pontos += 5

            if "dice" in texto.lower():
                pontos += 5

            if "sam" in texto.lower():
                pontos += 3

            if pontos >= 35:
                resultados.append((pontos, caminho, texto))

resultados.sort(key=lambda x: -x[0])

print("=== AVALIADORES ENCONTRADOS ===")

for pontos, caminho, _ in resultados[:12]:
    print(
        f"Pontuação={pontos:3d} | "
        f"{caminho.relative_to(ROOT)}"
    )

print("\n=== ESTRUTURA DOS MELHORES CANDIDATOS ===")

for pontos, caminho, texto in resultados[:3]:
    print("\n" + "=" * 65)
    print(caminho.relative_to(ROOT))
    print("=" * 65)

    linhas = texto.splitlines()

    padrao = re.compile(
        r"^\s*(def |class |from |import )|"
        r"DADOS\s*=|RUN\s*=|CHECKPOINT|"
        r"SUPORTE|QUERY|SAM|DICE|"
        r"image_size|NORMALIZ",
        re.IGNORECASE
    )

    encontrados = [
        (i, linha)
        for i, linha in enumerate(linhas, 1)
        if padrao.search(linha)
    ]

    for numero, linha in encontrados[:75]:
        print(f"{numero:04d}: {linha[:180]}")

print("\n=== RESULTADOS CT EXISTENTES ===")

PASTA_CT = (
    ROOT / "reproducao/"
    "eft_ct_felzenszwalb_fold0_10000_v1"
)

for nome in ALVOS:
    pasta = PASTA_CT / nome

    print(
        nome,
        "ENCONTRADO" if pasta.exists() else "NÃO ENCONTRADO"
    )

    if pasta.exists():
        for arquivo in sorted(pasta.glob("*.csv")):
            print("  ", arquivo.name)

print("\nInspeção concluída.")

In [ ]:
from pathlib import Path

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

SCRIPT = (
    ROOT
    / "exp/eft_ct_piloto200_v1/avaliar_alp_ct_10000.py"
)

assert SCRIPT.is_file(), f"Script não encontrado: {SCRIPT}"

linhas = SCRIPT.read_text(
    encoding="utf-8",
    errors="replace"
).splitlines()

print("=== AVALIADOR ALP DO CT ===")
print("Arquivo:", SCRIPT)
print("Total de linhas:", len(linhas))
print("=" * 70)

for numero, linha in enumerate(linhas, 1):
    print(f"{numero:04d}: {linha}")

print("\n=== CHECKPOINTS MRI DISPONÍVEIS ===")

PILOTO = (
    ROOT / "reproducao/eft_mri_chaost2_fold0_piloto200"
)

TREINO = (
    ROOT / "reproducao/eft_mri_chaost2_fold0_10000_v1"
)

for passo in [0, 200, 500, 1000, 2000, 5000, 10000]:
    if passo <= 200:
        caminho = PILOTO / f"checkpoint_{passo:04d}.pt"
    else:
        caminho = TREINO / f"checkpoint_{passo:05d}.pt"

    print(
        f"Passo {passo:5d}:",
        "DISPONÍVEL" if caminho.is_file() else "AUSENTE"
    )

    assert caminho.is_file(), caminho

print("\nInspeção concluída. Nenhum arquivo modificado.")

In [ ]:
from pathlib import Path
import subprocess
import hashlib
import json
import ast
import sys
import os

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

EXP = ROOT / "exp/eft_mri_chaos_v1"
DADOS = ROOT / "data/CHAOST2/chaos_MR_T2_normalized"

PILOTO = ROOT / "reproducao/eft_mri_chaost2_fold0_piloto200"
TREINO = ROOT / "reproducao/eft_mri_chaost2_fold0_10000_v1"

ORIGINAL = ROOT / "exp/eft_ct_piloto200_v1/avaliar_alp_ct_10000.py"
SCRIPT = EXP / "avaliar_alp_mri_10000.py"

SAIDA = TREINO / "avaliacao_alp_rins_positivas_f0_7ckpts"

LOG = SAIDA / "console.log"
LANCAMENTO = SAIDA / "lancamento.json"
CONFIG = SAIDA / "config_avaliacao.json"
CSV = SAIDA / "metricas_fatias.csv"
STATUS = SAIDA / "status.json"

CHECKPOINTS = [0, 200, 500, 1000, 2000, 5000, 10000]

assert ORIGINAL.is_file()
assert (DADOS / "classmap_1.json").is_file()

EXP.mkdir(parents=True, exist_ok=True)
SAIDA.mkdir(parents=True, exist_ok=True)

# ==========================================================
# 1. Verificar o protocolo de avaliação
# ==========================================================

classmap = json.loads(
    (DADOS / "classmap_1.json").read_text(encoding="utf-8")
)

QUERIES = list(range(4))
SUPORTE = 4
ORGAOS = [(2, "RK"), (3, "LK")]

fatias_por_orgao = {}

for classe, nome in ORGAOS:
    fatias_por_orgao[nome] = sum(
        len(classmap[nome][str(pid)])
        for pid in QUERIES
    )

total_por_checkpoint = sum(fatias_por_orgao.values())
total_inferencias = total_por_checkpoint * len(CHECKPOINTS)

assert total_por_checkpoint == 112, total_por_checkpoint
assert total_inferencias == 784, total_inferencias

for passo in CHECKPOINTS:
    if passo in (0, 200):
        caminho = PILOTO / f"checkpoint_{passo:04d}.pt"
    else:
        caminho = TREINO / f"checkpoint_{passo:05d}.pt"
    assert caminho.is_file(), caminho

print("=== PROTOCOLO MRI ===")
print("Suporte:", SUPORTE)
print("Queries:", QUERIES)
print("Fatias por órgão:", fatias_por_orgao)
print("Fatias por checkpoint:", total_por_checkpoint)
print("Checkpoints:", CHECKPOINTS)
print("Total de inferências:", total_inferencias)

# ==========================================================
# 2. Adaptar o avaliador CT
# ==========================================================

codigo = ORIGINAL.read_text(encoding="utf-8")

def substituir_exato(antigo, novo):
    global codigo

    quantidade = codigo.count(antigo)

    assert quantidade == 1, (
        f"Substituição não encontrada ou ambígua: "
        f"{antigo!r}, ocorrências={quantidade}"
    )

    codigo = codigo.replace(antigo, novo, 1)


substituir_exato(
    'DADOS = ROOT / "data/SABS/sabs_CT_normalized"',
    'DADOS = ROOT / "data/CHAOST2/chaos_MR_T2_normalized"'
)

substituir_exato(
    'TREINO = ROOT / "reproducao/eft_ct_felzenszwalb_fold0_10000_v1"',
    'TREINO = ROOT / "reproducao/eft_mri_chaost2_fold0_10000_v1"'
)

substituir_exato(
    'PILOTO = ROOT / "reproducao/eft_ct_felzenszwalb_fold0_piloto200"',
    'PILOTO = ROOT / "reproducao/eft_mri_chaost2_fold0_piloto200"'
)

substituir_exato(
    'ORGAOS = [(2, "KID_R"), (3, "KID_l")]',
    'ORGAOS = [(2, "RK"), (3, "LK")]'
)

substituir_exato(
    'QUERIES = list(range(6))',
    'QUERIES = list(range(4))'
)

substituir_exato(
    'SUPORTE = 6',
    'SUPORTE = 4'
)

substituir_exato(
    'assert total_esperado == 2975, total_esperado',
    'assert total_esperado == 784, total_esperado'
)

substituir_exato(
    'assert len(valores) == 6',
    'assert len(valores) == 4'
)

substituir_exato(
    'f"Suporte {nome}: paciente 6, fatia {z}"',
    'f"Suporte {nome}: paciente 4, fatia {z}"'
)

substituir_exato(
    'entre os seis pacientes',
    'entre os quatro pacientes'
)

# ==========================================================
# 3. Aplicar normalização MRI por volume
# ==========================================================

substituir_exato(
    '    return imagem, label',
    '''    imagem = imagem.astype(np.float32)

    media_volume = float(imagem.mean())
    desvio_volume = float(imagem.std())

    assert np.isfinite(media_volume)
    assert np.isfinite(desvio_volume)
    assert desvio_volume > 1e-6

    imagem = (imagem - media_volume) / desvio_volume

    return imagem, label'''
)

# Substituir apenas a função preparar_imagem.
# MRI já chega normalizada pelo volume.
inicio = codigo.index("def preparar_imagem(fatia):")
fim = codigo.index("def escrever_status(", inicio)

preparar_mri = '''def preparar_imagem(fatia):
    rgb = np.repeat(
        fatia.astype(np.float32)[..., None],
        3,
        axis=2
    )

    tensor = torch.from_numpy(
        np.ascontiguousarray(rgb)
    ).permute(2, 0, 1)[None].to(
        "cuda", dtype=torch.float32
    )

    assert tensor.shape == (1, 3, 256, 256)
    assert torch.isfinite(tensor).all().item()

    return tensor


'''

codigo = codigo[:inicio] + preparar_mri + codigo[fim:]

# ==========================================================
# 4. Validar a adaptação
# ==========================================================

ast.parse(codigo, filename=str(SCRIPT))

assert 'CHECKPOINTS = [0, 200, 500, 1000, 2000, 5000, 10000]' in codigo
assert 'SUPORTE = 4' in codigo
assert 'ORGAOS = [(2, "RK"), (3, "LK")]' in codigo
assert 'imagem = (imagem - media_volume) / desvio_volume' in codigo
assert 'modelo = construir_modelo("tiny", repo=ROOT)' in codigo
assert 'modelo.load_state_dict(' in codigo
assert '"dice": 2 * tp / max(2*tp+fp+fn, 1)' in codigo
assert 'np.std(valores, ddof=1)' in codigo
assert "KID_R" not in codigo
assert "KID_l" not in codigo

if SCRIPT.exists():
    assert SCRIPT.read_text(encoding="utf-8") == codigo, (
        "Script MRI existente difere do esperado."
    )
else:
    SCRIPT.write_text(codigo, encoding="utf-8")

sha = hashlib.sha256(codigo.encode("utf-8")).hexdigest()

config = {
    "dataset": "CHAOST2",
    "modalidade": "MRI T2-SPIR",
    "fold": 0,
    "encoder": "VMamba-Tiny",
    "avaliacao": "ALP sem SAM",
    "checkpoints": CHECKPOINTS,
    "suporte": SUPORTE,
    "queries": QUERIES,
    "orgaos": {"2": "RK", "3": "LK"},
    "normalizacao": "z-score por volume",
    "fatias_por_orgao": fatias_por_orgao,
    "total_inferencias": total_inferencias,
    "estatistica": "media por paciente e DP amostral entre pacientes",
    "script_sha256": sha
}

if CONFIG.exists():
    assert json.loads(
        CONFIG.read_text(encoding="utf-8")
    ) == config
else:
    temp = CONFIG.with_suffix(".json.tmp")
    temp.write_text(
        json.dumps(config, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    os.replace(temp, CONFIG)

print("\n=== SCRIPT DE AVALIAÇÃO ===")
print("Script:", SCRIPT)
print("SHA256:", sha)
print("Normalização: z-score por volume")
print("Métricas: Dice, IoU, precisão, recall")
print("Desvio padrão: entre os quatro pacientes")

# ==========================================================
# 5. Iniciar avaliação independente do Jupyter
# ==========================================================

def ler_json(caminho):
    if not caminho.exists():
        return {}
    try:
        return json.loads(caminho.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return {}

lancamento_anterior = ler_json(LANCAMENTO)
status = ler_json(STATUS)

pid_anterior = lancamento_anterior.get("pid")
ativo = False

if pid_anterior:
    try:
        proc = Path(f"/proc/{pid_anterior}")

        cmd = (proc / "cmdline").read_bytes()
        estado = (proc / "stat").read_text().split(") ")[1][0]

        ativo = (
            b"avaliar_alp_mri_10000.py" in cmd
            and estado not in ("Z", "X")
        )
    except (OSError, IndexError):
        pass

print("\n=== LANÇAMENTO ===")

if status.get("estado") == "concluido":
    print("Avaliação já concluída.")
    print("Nenhum novo processo iniciado.")

elif ativo:
    print("Avaliação já está em execução.")
    print("PID:", pid_anterior)

elif CSV.exists():
    print("Já existe um CSV de avaliação.")
    print("Nenhum processo iniciado para evitar sobrescrita.")
    print("Verifique o estado e o log antes de retomar.")

else:
    with LOG.open("a", encoding="utf-8") as log:
        processo = subprocess.Popen(
            [sys.executable, "-u", str(SCRIPT)],
            cwd=str(ROOT),
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True
        )

    registro = {
        "pid": processo.pid,
        "script": str(SCRIPT),
        "log": str(LOG),
        "total_inferencias": total_inferencias,
        "modo": "detached"
    }

    temp = LANCAMENTO.with_suffix(".json.tmp")
    temp.write_text(
        json.dumps(registro, indent=2),
        encoding="utf-8"
    )
    os.replace(temp, LANCAMENTO)

    print("AVALIAÇÃO ALP-MRI LANÇADA")
    print("PID:", processo.pid)
    print("Inferências previstas:", total_inferencias)
    print("Resultados:", SAIDA)
    print("Log:", LOG)
    print("Modo: detached")
    print("Lançamento solicitado; execução ainda não verificada.")

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict, deque
import csv
import json
import numpy as np

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

SAIDA = (
    ROOT
    / "reproducao/eft_mri_chaost2_fold0_10000_v1"
    / "avaliacao_alp_rins_positivas_f0_7ckpts"
)

DADOS = ROOT / "data/CHAOST2/chaos_MR_T2_normalized"

LANCAMENTO = SAIDA / "lancamento.json"
STATUS = SAIDA / "status.json"
CSV = SAIDA / "metricas_fatias.csv"
RESUMO = SAIDA / "resumo_pacientes.csv"
LOG = SAIDA / "console.log"

CHECKPOINTS = [0, 200, 500, 1000, 2000, 5000, 10000]
ORGAOS = ["RK", "LK"]
QUERIES = [0, 1, 2, 3]
TOTAL = 784

def ler_json(caminho):
    if not caminho.is_file():
        return {}
    try:
        return json.loads(caminho.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return {}

lancamento = ler_json(LANCAMENTO)
status = ler_json(STATUS)

# ==========================================================
# 1. Verificar processo
# ==========================================================

pid = lancamento.get("pid")
ativo = False

if pid:
    try:
        proc = Path(f"/proc/{pid}")
        cmd = (proc / "cmdline").read_bytes()
        estado = (proc / "stat").read_text().split(") ")[1][0]

        ativo = (
            b"avaliar_alp_mri_10000.py" in cmd
            and estado not in ("Z", "X")
        )
    except (OSError, IndexError):
        pass

# ==========================================================
# 2. Ler resultados já registrados
# ==========================================================

registros = []

if CSV.is_file():
    with CSV.open(
        newline="", encoding="utf-8"
    ) as arquivo:
        leitor = csv.DictReader(arquivo)

        for linha in leitor:
            try:
                registros.append({
                    "checkpoint": int(linha["checkpoint"]),
                    "orgao": linha["orgao"],
                    "paciente": int(linha["paciente"]),
                    "fatia": int(linha["fatia"]),
                    "dice": float(linha["dice"]),
                    "iou": float(linha["iou"]),
                    "precisao": float(linha["precisao"]),
                    "recall": float(linha["recall"]),
                    "pixels_pred": int(linha["pixels_pred"]),
                })
            except (ValueError, TypeError, KeyError):
                # Ignorar eventual linha incompleta durante escrita.
                continue

concluidas = len(registros)
percentual = min(100, 100 * concluidas / TOTAL)

print("=" * 66)
print("AVALIAÇÃO ALP-MRI — VMAMBA-TINY")
print("=" * 66)

print("PID:", pid)
print("Processo ativo:", "SIM" if ativo else "NÃO")
print("Estado:", status.get("estado", "inicializando"))

print(f"\nInferências: {concluidas}/{TOTAL}")
print(f"Progresso: {percentual:.2f}%")

blocos = int(percentual * 30 / 100)
print("[" + "█" * blocos + "░" * (30 - blocos) + "]")

# ==========================================================
# 3. Progresso por checkpoint
# ==========================================================

contagem = Counter(
    r["checkpoint"] for r in registros
)

print("\n=== PROGRESSO POR CHECKPOINT ===")
print(f"{'Passo':>8} | {'Inferências':>12} | Estado")
print("-" * 43)

for passo in CHECKPOINTS:
    n = contagem[passo]

    if n == 112:
        estado_cp = "CONCLUÍDO"
    elif n > 0:
        estado_cp = "em andamento"
    else:
        estado_cp = "pendente"

    print(
        f"{passo:8d} | "
        f"{n:5d}/112      | "
        f"{estado_cp}"
    )

# ==========================================================
# 4. Calcular Dice somente para grupos completos
# ==========================================================

classmap = ler_json(DADOS / "classmap_1.json")

agrupados = defaultdict(list)

for r in registros:
    chave = (
        r["checkpoint"],
        r["orgao"],
        r["paciente"]
    )
    agrupados[chave].append(r)

print("\n=== DICE MÉDIO ± DP ENTRE PACIENTES ===")
print("Exibindo apenas checkpoints integralmente avaliados.")
print()
print(
    f"{'Passo':>7} | "
    f"{'Rim direito':>20} | "
    f"{'Rim esquerdo':>20} | "
    f"{'Média':>7}"
)
print("-" * 67)

resultados_completos = []

for passo in CHECKPOINTS:
    if contagem[passo] != 112:
        continue

    medias_orgaos = {}
    desvios_orgaos = {}
    completo = True

    for orgao in ORGAOS:
        dice_pacientes = []

        for pid_query in QUERIES:
            chave = (passo, orgao, pid_query)
            valores = agrupados[chave]

            esperado = len(
                classmap[orgao][str(pid_query)]
            )

            if len(valores) != esperado:
                completo = False
                break

            dice_pacientes.append(
                float(np.mean([
                    r["dice"] for r in valores
                ]))
            )

        if not completo:
            break

        medias_orgaos[orgao] = float(
            np.mean(dice_pacientes)
        )

        desvios_orgaos[orgao] = float(
            np.std(dice_pacientes, ddof=1)
        )

    if not completo:
        print(
            f"Checkpoint {passo}: "
            "contagem por paciente inconsistente."
        )
        continue

    media_geral = (
        medias_orgaos["RK"] + medias_orgaos["LK"]
    ) / 2

    resultados_completos.append({
        "passo": passo,
        "media_geral": media_geral,
    })

    print(
        f"{passo:7d} | "
        f"{medias_orgaos['RK']:.4f} ± "
        f"{desvios_orgaos['RK']:.4f} | "
        f"{medias_orgaos['LK']:.4f} ± "
        f"{desvios_orgaos['LK']:.4f} | "
        f"{media_geral:.4f}"
    )

if not resultados_completos:
    print("Nenhum checkpoint completo ainda.")

# ==========================================================
# 5. Previsões vazias
# ==========================================================

print("\n=== PREVISÕES SEM FOREGROUND ===")

for passo in CHECKPOINTS:
    if contagem[passo] != 112:
        continue

    vazias = sum(
        1 for r in registros
        if r["checkpoint"] == passo
        and r["pixels_pred"] == 0
    )

    print(
        f"Checkpoint {passo:5d}: "
        f"{vazias}/112 previsões vazias"
    )

# ==========================================================
# 6. Diagnóstico
# ==========================================================

print("\n=== DIAGNÓSTICO ===")

if status.get("estado") == "erro":
    print("ERRO NA AVALIAÇÃO")
    print(status.get("traceback", "Sem traceback disponível."))

elif status.get("estado") == "concluido":
    if concluidas == TOTAL and RESUMO.is_file():
        print("AVALIAÇÃO CONCLUÍDA — 784 INFERÊNCIAS")
        print("Resumo por paciente: DISPONÍVEL")
    else:
        print("ATENÇÃO: conclusão registrada, mas faltam resultados.")

elif ativo:
    print("AVALIAÇÃO EM EXECUÇÃO")

else:
    print("Processo inativo sem conclusão confirmada.")

# ==========================================================
# 7. Log recente
# ==========================================================

print("\n=== ÚLTIMAS MENSAGENS DO LOG ===")

if LOG.is_file():
    with LOG.open(
        encoding="utf-8", errors="replace"
    ) as arquivo:
        for linha in deque(arquivo, maxlen=15):
            print(linha.rstrip())
else:
    print("Log ainda não disponível.")

print("\nArquivo de resultados:", CSV)

In [ ]:
from pathlib import Path

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

ARQUIVO = (
    ROOT
    / "exp/eft_ct_piloto200_v1/"
    "avaliar_alp_sam_ct_retomada.py"
)

assert ARQUIVO.is_file()

linhas = ARQUIVO.read_text(
    encoding="utf-8", errors="replace"
).splitlines()

def mostrar(titulo, inicio, fim):
    print("\n" + "=" * 65)
    print(titulo)
    print("=" * 65)

    for i in range(inicio - 1, min(fim, len(linhas))):
        print(f"{i+1:04d}: {linhas[i]}")

mostrar(
    "1. CONFIGURAÇÃO, NORMALIZAÇÃO E CHECKPOINTS",
    1, 110
)

mostrar(
    "2. SUPORTES E CONSTRUÇÃO DO PROTOSAM",
    170, 235
)

mostrar(
    "3. INFERÊNCIA ALP + SAM",
    275, 365
)

# Identificar como o ProtoSAM prepara imagens para SAM.
PROTO = ROOT / "models/ProtoSAM.py"

print("\n" + "=" * 65)
print("4. TRANSFORMAÇÕES DE ENTRADA NO PROTOSAM")
print("=" * 65)

assert PROTO.is_file()

fonte = PROTO.read_text(
    encoding="utf-8", errors="replace"
).splitlines()

termos = (
    "sam_transform",
    "sam_trans",
    "sam_preprocess",
    "sam_model.preprocess",
    "sam.image_encoder",
    "resize_longest",
    "query_image",
    "unnormalize",
    "image_mean",
    "image_std",
)

encontrados = [
    i for i, linha in enumerate(fonte)
    if any(t in linha.lower() for t in termos)
]

indices = set()

for i in encontrados:
    indices.update(
        range(max(0, i - 3), min(len(fonte), i + 5))
    )

for i in sorted(indices)[:180]:
    print(f"{i+1:04d}: {fonte[i]}")

print("\nInspeção concluída. Nenhum arquivo modificado.")

In [ ]:
from pathlib import Path
import subprocess
import json
import hashlib
import ast
import os
import sys

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
EXP = ROOT / "exp/eft_mri_chaos_v1"
TREINO = ROOT / "reproducao/eft_mri_chaost2_fold0_10000_v1"
PILOTO = ROOT / "reproducao/eft_mri_chaost2_fold0_piloto200"

SAIDA = TREINO / "avaliacao_alp_sam_rins_f0_7ckpts"
SCRIPT = EXP / "avaliar_alp_sam_mri.py"
LOG = SAIDA / "console.log"
LANCAMENTO = SAIDA / "lancamento.json"

REF_ALP = (
    TREINO / "avaliacao_alp_rins_positivas_f0_7ckpts"
    / "metricas_fatias.csv"
)

CFG_SAM = (
    ROOT / "reproducao"
    / "eft_tiny_passo_2000_validacao_20261011T221527472147Z"
    / "config.json"
)

assert REF_ALP.is_file()
assert CFG_SAM.is_file()
assert (ROOT / "pretrained_model/sam_vit_h.pth").is_file()

EXP.mkdir(parents=True, exist_ok=True)
SAIDA.mkdir(parents=True, exist_ok=True)

codigo = r'''
import os
import sys
import csv
import json
import time
import fcntl
import traceback
from pathlib import Path
from functools import lru_cache
from collections import defaultdict

import numpy as np
import torch
import SimpleITK as sitk

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
MODULOS = ROOT / "exp/eft_tiny_512_v1"
DADOS = ROOT / "data/CHAOST2/chaos_MR_T2_normalized"
TREINO = ROOT / "reproducao/eft_mri_chaost2_fold0_10000_v1"
PILOTO = ROOT / "reproducao/eft_mri_chaost2_fold0_piloto200"

SAIDA = TREINO / "avaliacao_alp_sam_rins_f0_7ckpts"
CSV_SAM = SAIDA / "metricas_fatias.csv"
CSV_PACIENTES = SAIDA / "resumo_pacientes.csv"
CSV_GERAL = SAIDA / "resumo_geral.csv"
STATUS = SAIDA / "status.json"

REF_ALP = (
    TREINO / "avaliacao_alp_rins_positivas_f0_7ckpts"
    / "metricas_fatias.csv"
)

CFG_SAM = (
    ROOT / "reproducao"
    / "eft_tiny_passo_2000_validacao_20261011T221527472147Z"
    / "config.json"
)

CHECKPOINTS = [0, 200, 500, 1000, 2000, 5000, 10000]
ORGAOS = [(2, "RK"), (3, "LK")]
QUERIES = [0, 1, 2, 3]
SUPORTE = 4
TOTAL = 784

CAMPOS = [
    "checkpoint", "orgao", "paciente", "fatia",
    "dice_alp", "dice_sam", "iou_sam",
    "precisao_sam", "recall_sam",
    "tp_sam", "fp_sam", "fn_sam",
    "pixels_gt", "pixels_sam", "sam_executado"
]

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(MODULOS))

from eft_modelo_672 import construir_modelo
from models.ProtoSAM import ProtoSAM, ALPNetWrapper, ALPNetInput

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.benchmark = False


def salvar_json(path, objeto):
    temporario = path.with_name(path.name + ".tmp")
    temporario.write_text(
        json.dumps(objeto, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    os.replace(temporario, path)


def checkpoint(passo):
    if passo in (0, 200):
        return PILOTO / f"checkpoint_{passo:04d}.pt"
    return TREINO / f"checkpoint_{passo:05d}.pt"


@lru_cache(maxsize=5)
def volume(pid):
    img = sitk.GetArrayFromImage(
        sitk.ReadImage(str(DADOS / f"image_{pid}.nii.gz"))
    ).astype(np.float32)

    gt = sitk.GetArrayFromImage(
        sitk.ReadImage(str(DADOS / f"label_{pid}.nii.gz"))
    )

    assert img.shape == gt.shape
    assert img.shape[1:] == (256, 256)
    assert np.isfinite(img).all()

    media = float(img.mean())
    desvio = float(img.std())

    assert np.isfinite(desvio) and desvio > 1e-6

    # Mesmo z-score por volume usado no treinamento
    # e na avaliação ALP-MRI anterior.
    img = (img - media) / desvio

    return img, gt


def preparar(fatia):
    rgb = np.repeat(
        fatia.astype(np.float32)[..., None],
        3, axis=2
    )

    tensor = torch.from_numpy(
        np.ascontiguousarray(rgb)
    ).permute(2, 0, 1)[None].to(
        "cuda", dtype=torch.float32
    )

    assert tensor.shape == (1, 3, 256, 256)
    return tensor


def metricas(pred, gt):
    tp = int((pred & gt).sum())
    fp = int((pred & ~gt).sum())
    fn = int((~pred & gt).sum())

    return {
        "dice": 2 * tp / max(2 * tp + fp + fn, 1),
        "iou": tp / max(tp + fp + fn, 1),
        "precisao": tp / max(tp + fp, 1),
        "recall": tp / max(tp + fn, 1),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "pixels_pred": int(pred.sum())
    }


def chave(registro):
    return (
        int(registro["checkpoint"]),
        registro["orgao"],
        int(registro["paciente"]),
        int(registro["fatia"])
    )


def resumir(registros):
    grupos = defaultdict(list)

    for r in registros:
        grupos[(
            int(r["checkpoint"]),
            r["orgao"],
            int(r["paciente"])
        )].append(r)

    pacientes = []

    for (passo, orgao, pid), valores in sorted(grupos.items()):
        alp = np.array(
            [float(r["dice_alp"]) for r in valores]
        )
        sam = np.array(
            [float(r["dice_sam"]) for r in valores]
        )

        pacientes.append({
            "checkpoint": passo,
            "orgao": orgao,
            "paciente": pid,
            "n_fatias": len(valores),
            "dice_alp_medio": float(alp.mean()),
            "dice_sam_medio": float(sam.mean()),
            "dice_sam_dp_fatias": (
                float(sam.std(ddof=1))
                if len(sam) > 1 else 0.0
            ),
            "delta_dice": float(sam.mean() - alp.mean()),
            "sem_prompt": sum(
                int(r["sam_executado"]) == 0
                for r in valores
            )
        })

    with CSV_PACIENTES.open(
        "w", newline="", encoding="utf-8"
    ) as f:
        writer = csv.DictWriter(
            f, fieldnames=list(pacientes[0])
        )
        writer.writeheader()
        writer.writerows(pacientes)

    geral = []

    print("\n=== DICE MÉDIO ± DP ENTRE PACIENTES ===")

    for passo in CHECKPOINTS:
        for _, orgao in ORGAOS:
            grupo = [
                p for p in pacientes
                if p["checkpoint"] == passo
                and p["orgao"] == orgao
            ]

            assert len(grupo) == 4

            alp = np.array([
                p["dice_alp_medio"] for p in grupo
            ])
            sam = np.array([
                p["dice_sam_medio"] for p in grupo
            ])

            registro = {
                "checkpoint": passo,
                "orgao": orgao,
                "n_pacientes": 4,
                "dice_alp_media": float(alp.mean()),
                "dice_alp_dp": float(alp.std(ddof=1)),
                "dice_sam_media": float(sam.mean()),
                "dice_sam_dp": float(sam.std(ddof=1)),
                "delta_dice": float(sam.mean() - alp.mean()),
                "sem_prompt": sum(p["sem_prompt"] for p in grupo)
            }

            geral.append(registro)

            print(
                f"{orgao} | passo {passo:5d} | "
                f"ALP={registro['dice_alp_media']:.4f} | "
                f"SAM={registro['dice_sam_media']:.4f} ± "
                f"{registro['dice_sam_dp']:.4f} | "
                f"Delta={registro['delta_dice']:+.4f}",
                flush=True
            )

    with CSV_GERAL.open(
        "w", newline="", encoding="utf-8"
    ) as f:
        writer = csv.DictWriter(
            f, fieldnames=list(geral[0])
        )
        writer.writeheader()
        writer.writerows(geral)


def main():
    assert torch.cuda.is_available()
    SAIDA.mkdir(parents=True, exist_ok=True)

    mapa = json.loads(
        (DADOS / "classmap_1.json").read_text()
    )

    # Construir a ordem completa e fixa de avaliação.
    esperadas = [
        (passo, nome, pid, int(z))
        for passo in CHECKPOINTS
        for _, nome in ORGAOS
        for pid in QUERIES
        for z in mapa[nome][str(pid)]
    ]

    assert len(esperadas) == TOTAL
    assert len(set(esperadas)) == TOTAL

    referencia = {}

    with REF_ALP.open(
        newline="", encoding="utf-8"
    ) as f:
        for r in csv.DictReader(f):
            k = chave(r)
            assert k not in referencia
            referencia[k] = r

    assert len(referencia) == TOTAL
    assert set(referencia) == set(esperadas)

    for passo in CHECKPOINTS:
        assert checkpoint(passo).is_file()

    # Retomada: aceitar apenas um prefixo completo
    # da ordem esperada, evitando duplicações.
    anteriores = []

    if CSV_SAM.exists():
        assert CSV_SAM.read_bytes().endswith(b"\n"), (
            "CSV interrompido no meio de uma linha. "
            "Requer verificação antes da retomada."
        )

        with CSV_SAM.open(
            newline="", encoding="utf-8"
        ) as f:
            reader = csv.DictReader(f)
            assert reader.fieldnames == CAMPOS
            anteriores = list(reader)

        assert len(anteriores) <= TOTAL

        for indice, registro in enumerate(anteriores):
            assert chave(registro) == esperadas[indice], (
                f"Ordem inconsistente na linha {indice+1}"
            )

    concluidos = len(anteriores)
    print(
        f"Retomada: {concluidos}/{TOTAL} inferências",
        flush=True
    )

    if concluidos == TOTAL:
        resumir(anteriores)
        salvar_json(STATUS, {
            "estado": "concluido",
            "concluidos": TOTAL,
            "total": TOTAL
        })
        return

    # Suporte fixo: paciente 4, fatia central positiva.
    img_s, gt_s = volume(SUPORTE)
    suportes = {}

    for classe, nome in ORGAOS:
        z_positivas = np.flatnonzero(
            np.any(gt_s == classe, axis=(1, 2))
        )

        assert len(z_positivas) > 0
        z = int(z_positivas[len(z_positivas) // 2])

        sup_img = preparar(img_s[z])
        sup_mask = torch.from_numpy(
            (gt_s[z] == classe).astype(np.float32)
        )[None].cuda()

        suportes[classe] = (sup_img, sup_mask)

        print(
            f"Suporte {nome}: paciente 4, fatia {z}",
            flush=True
        )

    cfg = dict(
        json.loads(CFG_SAM.read_text())["sam"]
    )

    cfg["image_size"] = [1024, 1024]
    cfg["alpnet_image_size"] = 256
    cfg["sam_pretrained_path"] = str(
        ROOT / "pretrained_model/sam_vit_h.pth"
    )

    assert cfg["use_bbox"] and cfg["use_points"]
    assert not cfg["use_mask"]

    salvar_json(SAIDA / "config_sam_utilizada.json", cfg)

    print("Construindo VMamba-Tiny + SAM-H...", flush=True)

    modelo = construir_modelo("tiny", repo=ROOT)
    modelo.cuda().eval().requires_grad_(False)

    protosam = ProtoSAM(
        coarse_segmentation_model=ALPNetWrapper(modelo),
        **cfg
    ).to("cuda", dtype=torch.float32)

    protosam.eval().requires_grad_(False)

    if not CSV_SAM.exists():
        with CSV_SAM.open(
            "x", newline="", encoding="utf-8"
        ) as f:
            csv.DictWriter(
                f, fieldnames=CAMPOS
            ).writeheader()

    inicio = time.perf_counter()
    checkpoint_atual = None

    salvar_json(STATUS, {
        "estado": "executando",
        "concluidos": concluidos,
        "total": TOTAL
    })

    with CSV_SAM.open(
        "a", newline="", encoding="utf-8", buffering=1
    ) as f:
        writer = csv.DictWriter(f, fieldnames=CAMPOS)

        for passo, nome, pid, z in esperadas[concluidos:]:
            if passo != checkpoint_atual:
                cp = torch.load(
                    checkpoint(passo),
                    map_location="cpu",
                    weights_only=False
                )

                assert int(cp["passo"]) == passo
                modelo.load_state_dict(
                    cp["modelo"], strict=True
                )

                del cp
                modelo.eval()
                protosam.eval()
                checkpoint_atual = passo

                print(
                    f"\nCHECKPOINT {passo}",
                    flush=True
                )

            classe = 2 if nome == "RK" else 3
            sup_img, sup_mask = suportes[classe]

            imagens, labels = volume(pid)
            gt = labels[z] == classe
            assert gt.any()

            query = preparar(imagens[z])

            entrada = ALPNetInput(
                support_images=[sup_img],
                support_labels=[sup_mask],
                query_images=query,
                isval=True,
                val_wsize=2,
                show_viz=False,
                supp_fts=None
            )

            with (
                torch.no_grad(),
                torch.autocast("cuda", enabled=False)
            ):
                pred_sam, scores = protosam(
                    query_image=query,
                    coarse_model_input=entrada,
                    degrees_rotate=0
                )

                pred_sam = (
                    pred_sam.detach()
                    .cpu().numpy().squeeze()
                )

            sam_executado = True

            # Mesmo tratamento do avaliador CT para
            # o retorno antecipado sem foreground.
            if pred_sam.shape == (1024, 1024):
                assert not np.any(pred_sam)
                pred_sam = np.zeros(
                    (256, 256), dtype=np.uint8
                )
                sam_executado = False

            assert pred_sam.shape == (256, 256)
            assert np.isin(pred_sam, [0, 1]).all()

            m = metricas(pred_sam.astype(bool), gt)
            ref = referencia[(passo, nome, pid, z)]

            registro = {
                "checkpoint": passo,
                "orgao": nome,
                "paciente": pid,
                "fatia": z,
                "dice_alp": float(ref["dice"]),
                "dice_sam": m["dice"],
                "iou_sam": m["iou"],
                "precisao_sam": m["precisao"],
                "recall_sam": m["recall"],
                "tp_sam": m["tp"],
                "fp_sam": m["fp"],
                "fn_sam": m["fn"],
                "pixels_gt": int(gt.sum()),
                "pixels_sam": m["pixels_pred"],
                "sam_executado": int(sam_executado)
            }

            writer.writerow(registro)
            concluidos += 1

            if concluidos % 10 == 0 or concluidos == TOTAL:
                salvar_json(STATUS, {
                    "estado": "executando",
                    "concluidos": concluidos,
                    "total": TOTAL,
                    "checkpoint": passo,
                    "orgao": nome,
                    "paciente": pid,
                    "tempo_decorrido_s": (
                        time.perf_counter() - inicio
                    )
                })

            if concluidos % 25 == 0:
                print(
                    f"{concluidos}/{TOTAL}",
                    flush=True
                )

            del query, entrada, pred_sam, scores

    with CSV_SAM.open(
        newline="", encoding="utf-8"
    ) as f:
        registros = list(csv.DictReader(f))

    assert len(registros) == TOTAL
    assert [chave(r) for r in registros] == esperadas

    resumir(registros)

    salvar_json(STATUS, {
        "estado": "concluido",
        "concluidos": TOTAL,
        "total": TOTAL,
        "tempo_total_s": time.perf_counter() - inicio
    })

    print("\nAVALIAÇÃO ALP + SAM MRI CONCLUÍDA", flush=True)


if __name__ == "__main__":
    SAIDA.mkdir(parents=True, exist_ok=True)

    with (SAIDA / "execucao.lock").open("w") as lock:
        fcntl.flock(
            lock.fileno(),
            fcntl.LOCK_EX | fcntl.LOCK_NB
        )

        try:
            main()
        except Exception:
            erro = traceback.format_exc()
            salvar_json(STATUS, {
                "estado": "erro",
                "traceback": erro
            })
            print(erro, flush=True)
            raise
'''

ast.parse(codigo, filename=str(SCRIPT))

if SCRIPT.exists():
    assert SCRIPT.read_text(encoding="utf-8") == codigo, (
        "Já existe um avaliador MRI diferente. "
        "Nenhum arquivo foi sobrescrito."
    )
else:
    SCRIPT.write_text(codigo, encoding="utf-8")

print("=== AVALIADOR ALP + SAM MRI ===")
print("Script:", SCRIPT)
print(
    "SHA256:",
    hashlib.sha256(codigo.encode()).hexdigest()
)
print("Checkpoints: 0, 200, 500, 1000, 2000, 5000, 10000")
print("Inferências previstas: 784")
print("Normalização do encoder: z-score por volume")
print("Refinamento: SAM-H com prompts ALP")

def ler_json(path):
    if not path.is_file():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return {}

status = ler_json(SAIDA / "status.json")
lancamento = ler_json(LANCAMENTO)

pid_anterior = lancamento.get("pid")
ativo = False

if pid_anterior:
    try:
        proc = Path(f"/proc/{pid_anterior}")
        cmd = (proc / "cmdline").read_bytes()
        estado = (proc / "stat").read_text().split(") ")[1][0]

        ativo = (
            b"avaliar_alp_sam_mri.py" in cmd
            and estado not in ("Z", "X")
        )
    except (OSError, IndexError):
        pass

print("\n=== EXECUÇÃO ===")

if status.get("estado") == "concluido":
    print("Avaliação já concluída. Nenhum processo iniciado.")
elif ativo:
    print("Avaliação já em execução. PID:", pid_anterior)
else:
    with LOG.open("a", encoding="utf-8") as log:
        processo = subprocess.Popen(
            [sys.executable, "-u", str(SCRIPT)],
            cwd=str(ROOT),
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True
        )

    registro = {
        "pid": processo.pid,
        "script": str(SCRIPT),
        "log": str(LOG),
        "total_inferencias": 784,
        "modo": "detached"
    }

    tmp = LANCAMENTO.with_suffix(".json.tmp")
    tmp.write_text(
        json.dumps(registro, indent=2),
        encoding="utf-8"
    )
    os.replace(tmp, LANCAMENTO)

    print("AVALIAÇÃO ALP + SAM LANÇADA")
    print("PID:", processo.pid)
    print("Resultados:", SAIDA)
    print("Log:", LOG)
    print("Lançamento solicitado; execução ainda não verificada.")

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict, deque
import csv
import json
import numpy as np

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

SAIDA = (
    ROOT / "reproducao"
    / "eft_mri_chaost2_fold0_10000_v1"
    / "avaliacao_alp_sam_rins_f0_7ckpts"
)

DADOS = ROOT / "data/CHAOST2/chaos_MR_T2_normalized"

LANCAMENTO = SAIDA / "lancamento.json"
STATUS = SAIDA / "status.json"
CSV = SAIDA / "metricas_fatias.csv"
RESUMO = SAIDA / "resumo_geral.csv"
LOG = SAIDA / "console.log"

CHECKPOINTS = [0, 200, 500, 1000, 2000, 5000, 10000]
ORGAOS = ["RK", "LK"]
QUERIES = [0, 1, 2, 3]
TOTAL = 784

def ler_json(path):
    if not path.is_file():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return {}

lancamento = ler_json(LANCAMENTO)
status = ler_json(STATUS)
classmap = ler_json(DADOS / "classmap_1.json")

# 1. Processo
pid = lancamento.get("pid")
ativo = False

if pid:
    try:
        proc = Path(f"/proc/{pid}")
        cmd = (proc / "cmdline").read_bytes()
        estado_proc = (
            (proc / "stat").read_text().split(") ")[1][0]
        )
        ativo = (
            b"avaliar_alp_sam_mri.py" in cmd
            and estado_proc not in ("Z", "X")
        )
    except (OSError, IndexError):
        pass

# 2. Resultados registrados
registros = []

if CSV.is_file():
    with CSV.open(newline="", encoding="utf-8") as f:
        for linha in csv.DictReader(f):
            try:
                registros.append({
                    "checkpoint": int(linha["checkpoint"]),
                    "orgao": linha["orgao"],
                    "paciente": int(linha["paciente"]),
                    "fatia": int(linha["fatia"]),
                    "dice_alp": float(linha["dice_alp"]),
                    "dice_sam": float(linha["dice_sam"]),
                    "iou_sam": float(linha["iou_sam"]),
                    "sam_executado": int(linha["sam_executado"])
                })
            except (ValueError, TypeError, KeyError):
                continue

concluidos = len(registros)
progresso = min(100, concluidos / TOTAL * 100)

print("=" * 68)
print("CHAOS MRI — AVALIAÇÃO ALP + SAM-H")
print("=" * 68)

print("PID:", pid)
print("Processo ativo:", "SIM" if ativo else "NÃO")
print("Estado:", status.get("estado", "inicializando"))
print(f"Inferências: {concluidos}/{TOTAL}")
print(f"Progresso: {progresso:.2f}%")

blocos = int(progresso * 30 / 100)
print("[" + "█" * blocos + "░" * (30 - blocos) + "]")

# 3. Progresso por checkpoint
contagem = Counter(r["checkpoint"] for r in registros)

print("\n=== PROGRESSO POR CHECKPOINT ===")
print(f"{'Passo':>8} | {'Inferências':>12} | Estado")
print("-" * 45)

for passo in CHECKPOINTS:
    n = contagem[passo]
    situacao = (
        "CONCLUÍDO" if n == 112
        else "em andamento" if n > 0
        else "pendente"
    )

    print(f"{passo:8d} | {n:5d}/112      | {situacao}")

# 4. Dice por paciente, evitando misturar fatias
agrupados = defaultdict(list)

for r in registros:
    agrupados[
        (r["checkpoint"], r["orgao"], r["paciente"])
    ].append(r)

print("\n=== DICE ALP versus ALP + SAM ===")
print("Média ± DP entre os quatro pacientes")
print()
print(
    f"{'Passo':>7} | {'Órgão':>5} | "
    f"{'ALP':>14} | {'SAM':>14} | {'Δ Dice':>8}"
)
print("-" * 65)

resumos = []

for passo in CHECKPOINTS:
    if contagem[passo] != 112:
        continue

    for orgao in ORGAOS:
        medias_alp = []
        medias_sam = []
        completo = True

        for pid_query in QUERIES:
            grupo = agrupados[(passo, orgao, pid_query)]
            esperado = len(classmap[orgao][str(pid_query)])

            if len(grupo) != esperado:
                completo = False
                break

            medias_alp.append(
                np.mean([r["dice_alp"] for r in grupo])
            )
            medias_sam.append(
                np.mean([r["dice_sam"] for r in grupo])
            )

        if not completo:
            print(f"{passo} {orgao}: dados incompletos")
            continue

        media_alp = float(np.mean(medias_alp))
        media_sam = float(np.mean(medias_sam))
        dp_alp = float(np.std(medias_alp, ddof=1))
        dp_sam = float(np.std(medias_sam, ddof=1))
        delta = media_sam - media_alp

        resumos.append({
            "passo": passo,
            "orgao": orgao,
            "alp": media_alp,
            "sam": media_sam,
            "delta": delta
        })

        print(
            f"{passo:7d} | "
            f"{orgao:>5} | "
            f"{media_alp:.4f} ± {dp_alp:.4f} | "
            f"{media_sam:.4f} ± {dp_sam:.4f} | "
            f"{delta:+.4f}"
        )

if not resumos:
    print("Aguardando conclusão do primeiro checkpoint.")

# 5. Média dos dois rins
print("\n=== MÉDIA DOS DOIS RINS ===")
print(f"{'Passo':>7} | {'ALP':>8} | {'SAM':>8} | {'Δ p.p.':>8}")
print("-" * 43)

for passo in CHECKPOINTS:
    grupo = [r for r in resumos if r["passo"] == passo]

    if len(grupo) != 2:
        continue

    alp = np.mean([r["alp"] for r in grupo])
    sam = np.mean([r["sam"] for r in grupo])

    print(
        f"{passo:7d} | {alp:8.4f} | "
        f"{sam:8.4f} | {(sam-alp)*100:+8.2f}"
    )

# 6. Ausência de prompts SAM
print("\n=== CASOS SEM EXECUÇÃO DO SAM ===")

for passo in CHECKPOINTS:
    if contagem[passo] != 112:
        continue

    sem_prompt = sum(
        r["sam_executado"] == 0
        for r in registros
        if r["checkpoint"] == passo
    )
    print(f"Checkpoint {passo:5d}: {sem_prompt}/112")

# 7. Diagnóstico
print("\n=== DIAGNÓSTICO ===")

if status.get("estado") == "erro":
    print("ERRO NA AVALIAÇÃO")
    print(status.get("traceback", "Sem detalhes."))

elif status.get("estado") == "concluido":
    if concluidos == TOTAL and RESUMO.is_file():
        print("AVALIAÇÃO CONCLUÍDA — 784 INFERÊNCIAS")
        print("Resumo geral: DISPONÍVEL")
    else:
        print("ATENÇÃO: conclusão incompleta.")

elif ativo:
    print("AVALIAÇÃO EM EXECUÇÃO")

else:
    print("Processo inativo sem conclusão confirmada.")

print("\n=== ÚLTIMAS MENSAGENS DO LOG ===")

if LOG.is_file():
    with LOG.open(
        encoding="utf-8", errors="replace"
    ) as f:
        for linha in deque(f, maxlen=15):
            print(linha.rstrip())
else:
    print("Log ainda não disponível.")

In [ ]:
from pathlib import Path
from collections import defaultdict
import csv
import json
import os
import numpy as np

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
BASE = ROOT / "reproducao/eft_mri_chaost2_fold0_10000_v1"

DIR_ALP = BASE / "avaliacao_alp_rins_positivas_f0_7ckpts"
DIR_SAM = BASE / "avaliacao_alp_sam_rins_f0_7ckpts"

CSV_ALP = DIR_ALP / "metricas_fatias.csv"
CSV_SAM = DIR_SAM / "metricas_fatias.csv"
SAIDA = DIR_SAM / "comparacao_pareada_pacientes.csv"

CHECKPOINTS = [0, 200, 500, 1000, 2000, 5000, 10000]
ORGAOS = ["RK", "LK"]
PACIENTES = [0, 1, 2, 3]

def ler_csv(caminho):
    with caminho.open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def chave(r):
    return (
        int(r["checkpoint"]),
        r["orgao"],
        int(r["paciente"]),
        int(r["fatia"])
    )

alp = ler_csv(CSV_ALP)
sam = ler_csv(CSV_SAM)

assert len(alp) == len(sam) == 784

mapa_alp = {chave(r): r for r in alp}
mapa_sam = {chave(r): r for r in sam}

assert len(mapa_alp) == len(mapa_sam) == 784
assert set(mapa_alp) == set(mapa_sam)

# Verificar que o ALP reutilizado pelo avaliador SAM
# corresponde exatamente ao resultado ALP original.
for k in mapa_alp:
    assert np.isclose(
        float(mapa_alp[k]["dice"]),
        float(mapa_sam[k]["dice_alp"]),
        atol=1e-8
    ), f"ALP divergente: {k}"

grupos = defaultdict(list)

for k, registro in mapa_sam.items():
    passo, orgao, paciente, fatia = k
    grupos[(passo, orgao, paciente)].append(registro)

resumo = []

for passo in CHECKPOINTS:
    for orgao in ORGAOS:
        for pid in PACIENTES:
            fatias = grupos[(passo, orgao, pid)]

            assert fatias

            dice_alp = float(np.mean([
                float(r["dice_alp"]) for r in fatias
            ]))

            dice_sam = float(np.mean([
                float(r["dice_sam"]) for r in fatias
            ]))

            resumo.append({
                "checkpoint": passo,
                "orgao": orgao,
                "paciente": pid,
                "n_fatias": len(fatias),
                "dice_alp": dice_alp,
                "dice_sam": dice_sam,
                "ganho_sam": dice_sam - dice_alp
            })

assert len(resumo) == 7 * 2 * 4

indice = {
    (r["checkpoint"], r["orgao"], r["paciente"]): r
    for r in resumo
}

for r in resumo:
    baseline = indice[
        (0, r["orgao"], r["paciente"])
    ]["dice_sam"]

    r["delta_eft_vs_inicial"] = r["dice_sam"] - baseline

print("=== 1. COMPARAÇÃO PAREADA COM O MODELO INICIAL ===")
print("Diferenças em pontos percentuais de Dice")
print()
print(
    f"{'Passo':>7} | "
    f"{'Δ médio':>9} | "
    f"{'DP pacientes':>12} | "
    f"{'Melhoraram':>11}"
)
print("-" * 52)

for passo in CHECKPOINTS:
    if passo == 0:
        continue

    # Uma observação por paciente:
    # média das diferenças dos dois rins.
    deltas = []

    for pid in PACIENTES:
        d = np.mean([
            indice[(passo, orgao, pid)]["delta_eft_vs_inicial"]
            for orgao in ORGAOS
        ])
        deltas.append(float(d))

    media = float(np.mean(deltas))
    dp = float(np.std(deltas, ddof=1))
    melhoraram = sum(d > 0 for d in deltas)

    print(
        f"{passo:7d} | "
        f"{media * 100:+8.2f} | "
        f"{dp * 100:11.2f} | "
        f"{melhoraram}/4"
    )

print("\n=== 2. DETALHAMENTO DOS CHECKPOINTS 5000 E 10000 ===")
print("Δ Dice SAM contra checkpoint 0, em p.p.")

for passo in (5000, 10000):
    print(f"\nCheckpoint {passo}")

    print(
        f"{'Paciente':>9} | "
        f"{'Rim direito':>12} | "
        f"{'Rim esquerdo':>13} | "
        f"{'Média':>9}"
    )
    print("-" * 55)

    for pid in PACIENTES:
        rk = indice[(passo, "RK", pid)]["delta_eft_vs_inicial"]
        lk = indice[(passo, "LK", pid)]["delta_eft_vs_inicial"]

        print(
            f"{pid:9d} | "
            f"{rk * 100:+11.2f} | "
            f"{lk * 100:+12.2f} | "
            f"{(rk + lk) * 50:+8.2f}"
        )

print("\n=== 3. SALVAR RESUMO REPRODUZÍVEL ===")

campos = list(resumo[0].keys())

temp = SAIDA.with_name(SAIDA.name + ".tmp")

with temp.open(
    "w", newline="", encoding="utf-8"
) as f:
    writer = csv.DictWriter(f, fieldnames=campos)
    writer.writeheader()
    writer.writerows(resumo)

if SAIDA.exists():
    assert SAIDA.read_bytes() == temp.read_bytes(), (
        "O resumo existente difere do novo cálculo."
    )
    temp.unlink()
    print("Resumo existente verificado.")
else:
    os.replace(temp, SAIDA)
    print("Resumo criado.")

print("Arquivo:", SAIDA)
print("Registros por paciente:", len(resumo))
print("\nANÁLISE PAREADA CONCLUÍDA")

In [ ]:
from pathlib import Path
import json
import re

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

print("=== 1. PESOS PRÉ-TREINADOS DISPONÍVEIS ===")

pastas = [
    ROOT / "pretrained_model",
    ROOT / "pretrained_models",
    Path.home() / ".cache/torch/hub/checkpoints",
]

for pasta in pastas:
    print(f"\n{pasta}")

    if not pasta.is_dir():
        print("  Não encontrada")
        continue

    arquivos = sorted(
        p for p in pasta.iterdir()
        if p.is_file()
    )

    for p in arquivos[:35]:
        print(
            f"  {p.name} "
            f"({p.stat().st_size / 1024**2:.1f} MiB)"
        )

    if len(arquivos) > 35:
        print(f"  ... mais {len(arquivos)-35} arquivos")


print("\n=== 2. REFERÊNCIAS AO DINO NO CÓDIGO ===")

padrao = re.compile(
    r"dinov2|dino_vit|vitl14|vit_l_14|"
    r"torch\.hub\.load|DINOv2",
    re.IGNORECASE
)

diretorios = [
    ROOT / "models",
    ROOT / "exp",
]

encontrados = []

for diretorio in diretorios:
    for arquivo in diretorio.rglob("*.py"):
        if any(
            parte in {"__pycache__", "source", "snapshots"}
            for parte in arquivo.parts
        ):
            continue

        if arquivo.stat().st_size > 500_000:
            continue

        linhas = arquivo.read_text(
            encoding="utf-8",
            errors="replace"
        ).splitlines()

        indices = [
            i for i, linha in enumerate(linhas)
            if padrao.search(linha)
        ]

        if indices:
            encontrados.append((arquivo, linhas, indices))

for arquivo, linhas, indices in encontrados[:12]:
    print(f"\n--- {arquivo.relative_to(ROOT)} ---")

    for i in indices[:12]:
        for j in range(max(0, i-2), min(len(linhas), i+3)):
            print(f"{j+1:04d}: {linhas[j]}")
        print("---")


print("\n=== 3. NOTEBOOK BASELINE ORIGINAL ===")

notebook = ROOT / "exp/00_baseline_original.ipynb"

if notebook.is_file():
    nb = json.loads(notebook.read_text(encoding="utf-8"))

    for i, celula in enumerate(nb["cells"]):
        if celula["cell_type"] != "code":
            continue

        fonte = "".join(celula.get("source", []))

        if not padrao.search(fonte):
            continue

        print(f"\nCélula {i}")

        linhas = fonte.splitlines()
        for linha in linhas[:45]:
            print(linha)

        if len(linhas) > 45:
            print(f"... mais {len(linhas)-45} linhas")
else:
    print("Notebook não encontrado.")


print("\n=== 4. CONFIGURAÇÕES DO MODELO ===")

candidatos = [
    ROOT / "exp/eft_tiny_512_v1/eft_modelo_672.py",
    ROOT / "models/ProtoSAM.py",
]

for arquivo in candidatos:
    if not arquivo.is_file():
        continue

    linhas = arquivo.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    print(f"\n--- {arquivo.relative_to(ROOT)} ---")

    for i, linha in enumerate(linhas):
        if (
            linha.lstrip().startswith(("def construir_modelo", "class "))
            or padrao.search(linha)
        ):
            print(f"{i+1:04d}: {linha[:180]}")

print("\nInspeção concluída. Nenhum arquivo modificado.")

In [ ]:
from pathlib import Path
import json

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

DINO_REPO = (
    ROOT / "pretrained_model/hub/facebookresearch_dinov2_main"
)

DINO_PESOS = (
    ROOT / "pretrained_model/hub/checkpoints/"
    "dinov2_vitl14_pretrain.pth"
)

print("=" * 65)
print("1. ARQUIVOS DO DINOV2-L")
print("=" * 65)

arquivos = {
    "Repositório DINOv2": DINO_REPO,
    "hubconf.py": DINO_REPO / "hubconf.py",
    "Pesos DINOv2-L": DINO_PESOS,
    "Pesos SAM-H": ROOT / "pretrained_model/sam_vit_h.pth",
}

for nome, caminho in arquivos.items():
    existe = caminho.exists()
    print(f"\n{nome}: {'OK' if existe else 'AUSENTE'}")
    print("  ", caminho)

    if caminho.is_file():
        print(
            "   Tamanho:",
            f"{caminho.stat().st_size / 1024**2:.1f} MiB"
        )

# Caso os pesos estejam em outra localização local,
# listar apenas candidatos conhecidos.
if not DINO_PESOS.is_file():
    print("\n=== BUSCA DE PESOS EM PASTAS LOCAIS ===")

    pastas = [
        ROOT / "pretrained_model",
        Path.home() / ".cache/torch/hub",
    ]

    for pasta in pastas:
        if not pasta.exists():
            continue

        for arquivo in pasta.rglob("*vitl14*.pth"):
            print(arquivo)

print("\n" + "=" * 65)
print("2. CONFIGURAÇÃO DO BASELINE ORIGINAL")
print("=" * 65)

NOTEBOOK = ROOT / "exp/00_baseline_original.ipynb"

assert NOTEBOOK.is_file(), NOTEBOOK

nb = json.loads(
    NOTEBOOK.read_text(encoding="utf-8")
)

for indice in (3, 26):
    celula = nb["cells"][indice]
    fonte = "".join(celula.get("source", []))

    print(f"\n--- CÉLULA ORIGINAL {indice} ---")

    linhas = fonte.splitlines()

    for numero, linha in enumerate(linhas[:115], 1):
        print(f"{numero:03d}: {linha}")

    if len(linhas) > 115:
        print(f"... {len(linhas)-115} linhas adicionais")

print("\n" + "=" * 65)
print("3. IMPLEMENTAÇÃO DO ENCODER")
print("=" * 65)

MODELO = ROOT / "models/grid_proto_fewshot.py"

linhas = MODELO.read_text(
    encoding="utf-8"
).splitlines()

for numero, linha in enumerate(linhas):
    if (
        linha.lstrip().startswith("def get_encoder(")
        or linha.lstrip().startswith("def get_features(")
        or linha.lstrip().startswith("def forward(")
    ):
        print(f"\n--- {linha.strip()} — linha {numero+1} ---")

        for i in range(
            numero,
            min(numero + 28, len(linhas))
        ):
            print(f"{i+1:04d}: {linhas[i]}")

print("\nINSPEÇÃO CONCLUÍDA — nenhum arquivo modificado.")